In [1]:
# ============================================================
# FRESH SERVER CUDA CHECK
# ============================================================

import torch

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    free, total = torch.cuda.mem_get_info()

    print(
        f"CUDA free : {free / 1024**3:.2f} GB"
    )

    print(
        f"CUDA total: {total / 1024**3:.2f} GB"
    )

[HAMI-core Msg(158:140162725371200:libvgpu.c:839)]: Initializing.....


CUDA available: True
CUDA free : 15.58 GB
CUDA total: 16.00 GB


[HAMI-core Msg(158:140162725371200:libvgpu.c:855)]: Initialized


In [2]:
# ============================================================
# UNFILTERED 1x AUGMENTATION EXPERIMENT
# PROJECT SETUP
# ============================================================

from pathlib import Path

import os
import gc
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

CLASSIFIER_DIR = (
    PROJECT_ROOT
    / "classifier"
)

FINE_TUNING_DIR = (
    PROJECT_ROOT
    / "fine tuning"
)

GENERATION_DIR = (
    FINE_TUNING_DIR
    / "outputs"
    / "synthetic_generation"
)


# ------------------------------------------------------------
# Original RUHSOLD dataset paths
# ------------------------------------------------------------

TRAIN_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_train.tsv"
)

VAL_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_validation.tsv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


# ------------------------------------------------------------
# Raw synthetic generation directories
# ------------------------------------------------------------

ROUND1_RAW_DIR = (
    GENERATION_DIR
    / "full"
)

ROUND2_RAW_DIR = (
    GENERATION_DIR
    / "full_round2"
)


# ------------------------------------------------------------
# Output directories for this experiment
# ------------------------------------------------------------

UNFILTERED_OUTPUT_DIR = (
    CLASSIFIER_DIR
    / "outputs"
    / "xlm_roberta_unfiltered_1x"
)

UNFILTERED_RESULTS_DIR = (
    CLASSIFIER_DIR
    / "outputs"
    / "xlm_roberta_unfiltered_1x_results"
)

UNFILTERED_DATA_DIR = (
    GENERATION_DIR
    / "unfiltered_control"
)


UNFILTERED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

UNFILTERED_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

UNFILTERED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MODEL / EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-base"

NUM_LABELS = 5
MAX_LENGTH = 128

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# FROZEN OPTUNA-SELECTED HYPERPARAMETERS
# ============================================================

BEST_LEARNING_RATE = (
    2.881129057462248e-05
)

BEST_BATCH_SIZE = 16

BEST_WEIGHT_DECAY = (
    0.03348739395854274
)

BEST_WARMUP_RATIO = (
    0.03756172525242821
)

FINAL_MAX_EPOCHS = 10

FINAL_EARLY_STOPPING_PATIENCE = 2


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: idx
    for idx, label
    in id2label.items()
}


# ============================================================
# BASIC PATH CHECKS
# ============================================================

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Training file exists:",
    TRAIN_PATH.exists()
)

print(
    "Validation file exists:",
    VAL_PATH.exists()
)

print(
    "Test file exists:",
    TEST_PATH.exists()
)

print(
    "Round 1 raw dir exists:",
    ROUND1_RAW_DIR.exists()
)

print(
    "Round 2 raw dir exists:",
    ROUND2_RAW_DIR.exists()
)

print(
    "Unfiltered results dir:",
    UNFILTERED_RESULTS_DIR
)

Project root: /home/jovyan/project work/data_analyssis
Training file exists: True
Validation file exists: True
Test file exists: True
Round 1 raw dir exists: True
Round 2 raw dir exists: True
Unfiltered results dir: /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_results


In [3]:
# ============================================================
# LOAD ORIGINAL RUHSOLD SPLITS
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ],
)

val_df = pd.read_csv(
    VAL_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ],
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    names=[
        "tweet",
        "label",
    ],
)


print(
    "Training samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(val_df)
)

print(
    "Test samples:",
    len(test_df)
)


print(
    "\nTraining class distribution:"
)

display(
    train_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

Training samples: 6408
Validation samples: 801
Test samples: 2003

Training class distribution:


,label,count
0,0,1537
1,1,3423
2,2,500
3,3,537
4,4,411


In [4]:
# ============================================================
# VERIFY ORIGINAL DATASET
# ============================================================

expected_labels = {
    0,
    1,
    2,
    3,
    4,
}


assert (
    set(
        train_df[
            "label"
        ].unique()
    )
    ==
    expected_labels
)

assert (
    set(
        val_df[
            "label"
        ].unique()
    )
    ==
    expected_labels
)

assert (
    set(
        test_df[
            "label"
        ].unique()
    )
    ==
    expected_labels
)


assert (
    train_df[
        "tweet"
    ]
    .notna()
    .all()
)

assert (
    val_df[
        "tweet"
    ]
    .notna()
    .all()
)

assert (
    test_df[
        "tweet"
    ]
    .notna()
    .all()
)


print(
    "Original RUHSOLD splits verified successfully."
)

Original RUHSOLD splits verified successfully.


In [5]:
# ============================================================
# LOAD RAW ROUND 1 + ROUND 2 SYNTHETIC GENERATIONS
# ============================================================

round1_files = sorted(
    ROUND1_RAW_DIR.glob("*.csv")
)

round2_files = sorted(
    ROUND2_RAW_DIR.glob("*.csv")
)

print(
    "Round 1 files found:",
    len(round1_files)
)

print(
    "Round 2 files found:",
    len(round2_files)
)


if len(round1_files) == 0:
    raise RuntimeError(
        "No Round 1 raw generation files found."
    )

if len(round2_files) == 0:
    raise RuntimeError(
        "No Round 2 raw generation files found."
    )


# ============================================================
# COMBINE ALL RAW GENERATION FILES
# ============================================================

raw_frames = []


for path in round1_files:

    batch_df = pd.read_csv(
        path
    )

    batch_df[
        "generation_round"
    ] = 1

    batch_df[
        "source_batch_file"
    ] = path.name

    raw_frames.append(
        batch_df
    )


for path in round2_files:

    batch_df = pd.read_csv(
        path
    )

    batch_df[
        "generation_round"
    ] = 2

    batch_df[
        "source_batch_file"
    ] = path.name

    raw_frames.append(
        batch_df
    )


raw_synthetic_pool_df = pd.concat(
    raw_frames,
    ignore_index=True,
)


# ============================================================
# VERIFY RAW POOL
# ============================================================

print(
    "\nCombined raw synthetic pool shape:",
    raw_synthetic_pool_df.shape
)

print(
    "\nRaw class distribution:"
)

display(
    raw_synthetic_pool_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)


print(
    "\nMissing generated texts:",
    raw_synthetic_pool_df[
        "generated_text"
    ]
    .isna()
    .sum()
)

print(
    "Duplicate production candidate IDs:",
    raw_synthetic_pool_df[
        "production_candidate_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Exact duplicate generated texts:",
    raw_synthetic_pool_df[
        "generated_text"
    ]
    .duplicated()
    .sum()
)


# ============================================================
# EXPECTED RAW COUNTS
# ============================================================

expected_raw_counts = {
    2: 1900,
    3: 2100,
    4: 1050,
}


actual_raw_counts = (
    raw_synthetic_pool_df[
        "class_id"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\nExpected raw counts:"
)

print(
    expected_raw_counts
)

print(
    "\nActual raw counts:"
)

print(
    actual_raw_counts
)


assert len(
    raw_synthetic_pool_df
) == 5050

assert (
    actual_raw_counts
    ==
    expected_raw_counts
)

assert (
    raw_synthetic_pool_df[
        "production_candidate_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    raw_synthetic_pool_df[
        "generated_text"
    ]
    .isna()
    .sum()
    == 0
)


print(
    "\nRaw unfiltered synthetic pool verified successfully."
)

Round 1 files found: 12
Round 2 files found: 15

Combined raw synthetic pool shape: (5050, 18)

Raw class distribution:


,class_id,target_label,count
0,3,Sexism,2100
1,2,Religious Hate,1900
2,4,Profane,1050



Missing generated texts: 0
Duplicate production candidate IDs: 0
Exact duplicate generated texts: 70

Expected raw counts:
{2: 1900, 3: 2100, 4: 1050}

Actual raw counts:
{2: 1900, 3: 2100, 4: 1050}

Raw unfiltered synthetic pool verified successfully.


In [6]:
# ============================================================
# BUILD MATCHED 1x UNFILTERED SYNTHETIC SET
# ============================================================

UNFILTERED_TARGET_COUNTS = {
    2: 500,   # Religious Hate
    3: 537,   # Sexism
    4: 411,   # Profane
}

UNFILTERED_SELECTION_SEED = 42


unfiltered_parts = []


for class_id, target_count in (
    UNFILTERED_TARGET_COUNTS.items()
):

    class_df = (
        raw_synthetic_pool_df[
            raw_synthetic_pool_df[
                "class_id"
            ].astype(int) == class_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"{class_id} - {id2label[class_id]}"
    )

    print(
        f"Available: {len(class_df)}"
    )

    print(
        f"Required : {target_count}"
    )

    selected_df = (
        class_df
        .sample(
            n=target_count,
            random_state=UNFILTERED_SELECTION_SEED,
            replace=False,
        )
        .copy()
        .reset_index(drop=True)
    )

    unfiltered_parts.append(
        selected_df
    )


# ============================================================
# COMBINE SELECTED RAW SYNTHETIC SAMPLES
# ============================================================

unfiltered_1x_df = pd.concat(
    unfiltered_parts,
    ignore_index=True,
)


print(
    "\nUnfiltered 1x synthetic shape:",
    unfiltered_1x_df.shape
)

print(
    "\nClass distribution:"
)

display(
    unfiltered_1x_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(
    unfiltered_1x_df
) == 1448

for class_id, expected_count in (
    UNFILTERED_TARGET_COUNTS.items()
):

    actual_count = int(
        (
            unfiltered_1x_df[
                "class_id"
            ].astype(int)
            == class_id
        ).sum()
    )

    assert actual_count == expected_count


print(
    "\nMatched unfiltered 1x dataset created successfully."
)

2 - Religious Hate
Available: 1900
Required : 500
3 - Sexism
Available: 2100
Required : 537
4 - Profane
Available: 1050
Required : 411

Unfiltered 1x synthetic shape: (1448, 18)

Class distribution:


,class_id,target_label,count
0,3,Sexism,537
1,2,Religious Hate,500
2,4,Profane,411



Matched unfiltered 1x dataset created successfully.


In [7]:
# ============================================================
# SAVE MATCHED UNFILTERED 1x SYNTHETIC DATASET
# ============================================================

UNFILTERED_1X_PATH = (
    UNFILTERED_DATA_DIR
    / "unfiltered_1x_synthetic_training_set.csv"
)

unfiltered_1x_df.to_csv(
    UNFILTERED_1X_PATH,
    index=False,
    encoding="utf-8",
)

print(
    "Saved unfiltered 1x dataset to:"
)

print(
    UNFILTERED_1X_PATH
)

Saved unfiltered 1x dataset to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/unfiltered_control/unfiltered_1x_synthetic_training_set.csv


In [8]:
# ============================================================
# CONVERT UNFILTERED SYNTHETIC DATA TO CLASSIFIER FORMAT
# ============================================================

unfiltered_classifier_df = (
    unfiltered_1x_df[
        [
            "generated_text",
            "class_id",
        ]
    ]
    .copy()
    .rename(
        columns={
            "generated_text": "tweet",
            "class_id": "label",
        }
    )
)

unfiltered_classifier_df[
    "label"
] = (
    unfiltered_classifier_df[
        "label"
    ].astype(int)
)


print(
    "Unfiltered classifier dataset:",
    unfiltered_classifier_df.shape
)

print(
    "\nClass distribution:"
)

display(
    unfiltered_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)


assert len(
    unfiltered_classifier_df
) == 1448

assert (
    unfiltered_classifier_df[
        "tweet"
    ]
    .notna()
    .all()
)

print(
    "\nUnfiltered classifier dataset verified."
)

Unfiltered classifier dataset: (1448, 2)

Class distribution:


,label,count
0,2,500
1,3,537
2,4,411



Unfiltered classifier dataset verified.


In [9]:
# ============================================================
# EXACT-TEXT LEAKAGE CHECK
# ============================================================

unfiltered_text_set = set(
    unfiltered_classifier_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

val_text_set = set(
    val_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)

test_text_set = set(
    test_df[
        "tweet"
    ]
    .astype(str)
    .str.strip()
)


unfiltered_val_overlap = (
    unfiltered_text_set
    &
    val_text_set
)

unfiltered_test_overlap = (
    unfiltered_text_set
    &
    test_text_set
)


print(
    "Unfiltered-validation exact overlap:",
    len(unfiltered_val_overlap)
)

print(
    "Unfiltered-test exact overlap:",
    len(unfiltered_test_overlap)
)

Unfiltered-validation exact overlap: 1
Unfiltered-test exact overlap: 1


In [10]:
# ============================================================
# INSPECT EXACT VALIDATION / TEST OVERLAPS
# ============================================================

print("=" * 70)
print("VALIDATION OVERLAP")
print("=" * 70)

validation_overlap_df = (
    unfiltered_1x_df[
        unfiltered_1x_df[
            "generated_text"
        ]
        .astype(str)
        .str.strip()
        .isin(
            unfiltered_val_overlap
        )
    ]
    .copy()
)

display(
    validation_overlap_df[
        [
            "production_candidate_id",
            "class_id",
            "target_label",
            "generated_text",
            "generation_round",
            "source_batch_file",
        ]
    ]
)


print("\n" + "=" * 70)
print("TEST OVERLAP")
print("=" * 70)

test_overlap_df = (
    unfiltered_1x_df[
        unfiltered_1x_df[
            "generated_text"
        ]
        .astype(str)
        .str.strip()
        .isin(
            unfiltered_test_overlap
        )
    ]
    .copy()
)

display(
    test_overlap_df[
        [
            "production_candidate_id",
            "class_id",
            "target_label",
            "generated_text",
            "generation_round",
            "source_batch_file",
        ]
    ]
)

VALIDATION OVERLAP


,production_candidate_id,class_id,target_label,generated_text,generation_round,source_batch_file
1421,class4_batch003_sample030501,4,Profane,bhenchod,1,profane_batch_003.csv



TEST OVERLAP


,production_candidate_id,class_id,target_label,generated_text,generation_round,source_batch_file
1037,class4_batch002_sample030352,4,Profane,randi ki nasal,1,profane_batch_002.csv


In [11]:
# ============================================================
# REMOVE EVALUATION LEAKAGE AND REPLACE WITH RAW SAME-CLASS
# SAMPLES
# ============================================================

LEAKAGE_REPLACEMENT_SEED = 42


# ------------------------------------------------------------
# Create set containing all validation + test texts
# ------------------------------------------------------------

evaluation_text_set = (
    val_text_set
    |
    test_text_set
)


# ------------------------------------------------------------
# Identify selected synthetic rows that leak
# ------------------------------------------------------------

selected_normalized_text = (
    unfiltered_1x_df[
        "generated_text"
    ]
    .astype(str)
    .str.strip()
)


leakage_mask = (
    selected_normalized_text
    .isin(
        evaluation_text_set
    )
)


leaked_rows_df = (
    unfiltered_1x_df[
        leakage_mask
    ]
    .copy()
)


print(
    "Leaked selected rows:",
    len(leaked_rows_df)
)

display(
    leaked_rows_df[
        [
            "production_candidate_id",
            "class_id",
            "target_label",
            "generated_text",
        ]
    ]
)


# ------------------------------------------------------------
# Start with only non-leaking selected samples
# ------------------------------------------------------------

clean_unfiltered_1x_df = (
    unfiltered_1x_df[
        ~leakage_mask
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Candidate IDs already selected
# ------------------------------------------------------------

already_selected_ids = set(
    clean_unfiltered_1x_df[
        "production_candidate_id"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# Replace leaked samples CLASS BY CLASS
# ------------------------------------------------------------

replacement_rows = []


for class_id in sorted(
    leaked_rows_df[
        "class_id"
    ]
    .astype(int)
    .unique()
):

    number_needed = int(
        (
            leaked_rows_df[
                "class_id"
            ].astype(int)
            == class_id
        ).sum()
    )

    print(
        f"\nClass {class_id} - "
        f"{id2label[class_id]}: "
        f"need {number_needed} replacement(s)"
    )

    # --------------------------------------------------------
    # Remaining raw samples from the SAME class
    # --------------------------------------------------------

    candidate_pool = (
        raw_synthetic_pool_df[
            raw_synthetic_pool_df[
                "class_id"
            ].astype(int)
            == class_id
        ]
        .copy()
    )

    # Remove candidates already selected.
    candidate_pool = (
        candidate_pool[
            ~candidate_pool[
                "production_candidate_id"
            ]
            .astype(str)
            .isin(
                already_selected_ids
            )
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Remove validation/test exact overlaps ONLY
    # --------------------------------------------------------

    candidate_pool[
        "_normalized_text"
    ] = (
        candidate_pool[
            "generated_text"
        ]
        .astype(str)
        .str.strip()
    )

    candidate_pool = (
        candidate_pool[
            ~candidate_pool[
                "_normalized_text"
            ]
            .isin(
                evaluation_text_set
            )
        ]
        .copy()
    )

    candidate_pool = (
        candidate_pool.drop(
            columns=[
                "_normalized_text"
            ]
        )
    )

    if len(candidate_pool) < number_needed:

        raise RuntimeError(
            f"Not enough valid raw replacements "
            f"for class {class_id}."
        )

    # --------------------------------------------------------
    # Deterministic replacement sampling
    # --------------------------------------------------------

    class_replacements = (
        candidate_pool
        .sample(
            n=number_needed,
            random_state=(
                LEAKAGE_REPLACEMENT_SEED
                + class_id
            ),
            replace=False,
        )
        .copy()
    )

    replacement_rows.append(
        class_replacements
    )

    already_selected_ids.update(
        class_replacements[
            "production_candidate_id"
        ]
        .astype(str)
        .tolist()
    )


# ============================================================
# ADD REPLACEMENTS
# ============================================================

replacement_df = pd.concat(
    replacement_rows,
    ignore_index=True,
)


unfiltered_1x_df = pd.concat(
    [
        clean_unfiltered_1x_df,
        replacement_df,
    ],
    ignore_index=True,
)


print(
    "\nFinal unfiltered 1x size:",
    len(unfiltered_1x_df)
)

Leaked selected rows: 2


,production_candidate_id,class_id,target_label,generated_text
1037,class4_batch002_sample030352,4,Profane,randi ki nasal
1421,class4_batch003_sample030501,4,Profane,bhenchod



Class 4 - Profane: need 2 replacement(s)

Final unfiltered 1x size: 1448


In [12]:
# ============================================================
# VERIFY MATCHED CLASS COUNTS AFTER REPLACEMENT
# ============================================================

print(
    "\nClass distribution after leakage replacement:"
)

display(
    unfiltered_1x_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)


assert len(
    unfiltered_1x_df
) == 1448


expected_unfiltered_counts = {
    2: 500,
    3: 537,
    4: 411,
}


actual_unfiltered_counts = (
    unfiltered_1x_df[
        "class_id"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_unfiltered_counts
    ==
    expected_unfiltered_counts
)


print(
    "Matched class counts preserved."
)


Class distribution after leakage replacement:


,class_id,target_label,count
0,3,Sexism,537
1,2,Religious Hate,500
2,4,Profane,411


Matched class counts preserved.


In [13]:
# ============================================================
# FINAL LEAKAGE CHECK
# ============================================================

final_unfiltered_text_set = set(
    unfiltered_1x_df[
        "generated_text"
    ]
    .astype(str)
    .str.strip()
)


final_val_overlap = (
    final_unfiltered_text_set
    &
    val_text_set
)

final_test_overlap = (
    final_unfiltered_text_set
    &
    test_text_set
)


print(
    "Final validation overlap:",
    len(final_val_overlap)
)

print(
    "Final test overlap:",
    len(final_test_overlap)
)


assert len(
    final_val_overlap
) == 0

assert len(
    final_test_overlap
) == 0


print(
    "\n✓ Evaluation leakage removed successfully."
)

Final validation overlap: 0
Final test overlap: 0

✓ Evaluation leakage removed successfully.


In [14]:
unfiltered_1x_df.to_csv(
    UNFILTERED_1X_PATH,
    index=False,
    encoding="utf-8",
)

print(
    "Corrected frozen unfiltered 1x dataset saved."
)

Corrected frozen unfiltered 1x dataset saved.


In [46]:
# ============================================================
# PREPARE FINAL UNFILTERED 1x CLASSIFIER DATA
# ============================================================

# ============================================================
# CHECKPOINT SELECTION METRIC
# ============================================================

CHECKPOINT_SELECTION_METRIC = "macro_f1"

assert CHECKPOINT_SELECTION_METRIC == "macro_f1"

print(
    "Checkpoint selection metric:",
    CHECKPOINT_SELECTION_METRIC
)
unfiltered_classifier_df = (
    unfiltered_1x_df[
        [
            "generated_text",
            "class_id",
        ]
    ]
    .copy()
    .rename(
        columns={
            "generated_text": "tweet",
            "class_id": "label",
        }
    )
)

unfiltered_classifier_df[
    "label"
] = (
    unfiltered_classifier_df[
        "label"
    ].astype(int)
)


# ============================================================
# VERIFY SYNTHETIC CLASSIFIER DATA
# ============================================================

print(
    "Unfiltered synthetic classifier dataset:",
    unfiltered_classifier_df.shape
)

print(
    "\nSynthetic class distribution:"
)

display(
    unfiltered_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)


assert len(
    unfiltered_classifier_df
) == 1448

assert (
    unfiltered_classifier_df[
        "tweet"
    ]
    .notna()
    .all()
)

assert (
    unfiltered_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    {
        2: 500,
        3: 537,
        4: 411,
    }
)


print(
    "\nFinal unfiltered synthetic classifier dataset verified."
)

Checkpoint selection metric: macro_f1
Unfiltered synthetic classifier dataset: (1448, 2)

Synthetic class distribution:


,label,count
0,2,500
1,3,537
2,4,411



Final unfiltered synthetic classifier dataset verified.


In [47]:
# ============================================================
# BUILD + VERIFY ORIGINAL + UNFILTERED 1x TRAINING DATA
#
# IMPORTANT:
# - Uses the frozen unfiltered synthetic set
# - Builds the augmented dataframe ONCE
# - The same dataframe must be reused for seeds 42, 43, 44
# ============================================================


# ============================================================
# BASIC SYNTHETIC DATA CHECKS
# ============================================================

assert (
    len(unfiltered_classifier_df)
    ==
    SYNTHETIC_1X_SIZE
), (
    f"Unexpected synthetic size.\n"
    f"Expected: {SYNTHETIC_1X_SIZE}\n"
    f"Actual:   {len(unfiltered_classifier_df)}"
)


assert (
    unfiltered_classifier_df["tweet"]
    .isna()
    .sum()
    ==
    0
)


assert (
    unfiltered_classifier_df["label"]
    .isna()
    .sum()
    ==
    0
)


assert (
    set(
        unfiltered_classifier_df[
            "label"
        ]
        .astype(int)
        .unique()
    )
    ==
    {
        2,
        3,
        4,
    }
)


# ============================================================
# VERIFY SYNTHETIC CLASS COUNTS
# ============================================================

actual_synthetic_counts = (
    unfiltered_classifier_df[
        "label"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_synthetic_counts
    ==
    UNFILTERED_TARGET_COUNTS
), (
    f"Unexpected synthetic class distribution.\n"
    f"Expected: {UNFILTERED_TARGET_COUNTS}\n"
    f"Actual:   {actual_synthetic_counts}"
)


# ============================================================
# BUILD ORIGINAL + UNFILTERED 1x TRAINING DATA
#
# IMPORTANT:
# This dataframe is created ONCE before the seed loop.
# It must NOT be rebuilt, resampled, or shuffled separately
# for individual seeds.
# ============================================================

train_unfiltered_aug_df = pd.concat(
    [
        train_df[
            [
                "tweet",
                "label",
            ]
        ].copy(),

        unfiltered_classifier_df[
            [
                "tweet",
                "label",
            ]
        ].copy(),
    ],
    ignore_index=True,
)


# ============================================================
# NORMALIZE TYPES
# ============================================================

train_unfiltered_aug_df[
    "tweet"
] = (
    train_unfiltered_aug_df[
        "tweet"
    ]
    .astype(str)
)


train_unfiltered_aug_df[
    "label"
] = (
    train_unfiltered_aug_df[
        "label"
    ]
    .astype(int)
)


# ============================================================
# VERIFY FINAL TRAINING DATA SIZE
# ============================================================

assert (
    len(train_df)
    ==
    ORIGINAL_TRAIN_SIZE
)


assert (
    len(
        unfiltered_classifier_df
    )
    ==
    SYNTHETIC_1X_SIZE
)


assert (
    len(
        train_unfiltered_aug_df
    )
    ==
    FINAL_UNFILTERED_TRAIN_SIZE
)


# ============================================================
# VERIFY FINAL CLASS DISTRIBUTION
# ============================================================

actual_augmented_counts = (
    train_unfiltered_aug_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_augmented_counts
    ==
    EXPECTED_UNFILTERED_AUGMENTED_COUNTS
), (
    f"Unexpected augmented class distribution.\n"
    f"Expected: {EXPECTED_UNFILTERED_AUGMENTED_COUNTS}\n"
    f"Actual:   {actual_augmented_counts}"
)


# ============================================================
# VERIFY NO MISSING VALUES
# ============================================================

assert (
    train_unfiltered_aug_df[
        "tweet"
    ]
    .isna()
    .sum()
    ==
    0
)


assert (
    train_unfiltered_aug_df[
        "label"
    ]
    .isna()
    .sum()
    ==
    0
)


# ============================================================
# OPTIONAL BUT USEFUL:
# SAVE A HASH OF THE FIXED TRAINING DATA
#
# This gives us evidence that the exact same dataframe is
# reused across all three seeds.
# ============================================================

import hashlib


def dataframe_sha256(df):

    canonical_text = (
        df[
            [
                "tweet",
                "label",
            ]
        ]
        .astype(
            {
                "tweet": str,
                "label": int,
            }
        )
        .to_csv(
            index=False,
            lineterminator="\n",
        )
    )

    return hashlib.sha256(
        canonical_text.encode(
            "utf-8"
        )
    ).hexdigest()


UNFILTERED_TRAINING_DATA_HASH = (
    dataframe_sha256(
        train_unfiltered_aug_df
    )
)


# ============================================================
# DISPLAY VERIFICATION
# ============================================================

print("=" * 70)
print(
    "UNFILTERED 1x AUGMENTED TRAINING DATA VERIFICATION"
)
print("=" * 70)


print(
    "Original RUHSOLD training samples:",
    len(train_df)
)

print(
    "Unfiltered synthetic samples:",
    len(
        unfiltered_classifier_df
    )
)

print(
    "Final augmented training samples:",
    len(
        train_unfiltered_aug_df
    )
)


print(
    "\nSynthetic class distribution:"
)

display(
    unfiltered_classifier_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nFinal augmented class distribution:"
)

display(
    train_unfiltered_aug_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nFixed augmented training-data SHA256:"
)

print(
    UNFILTERED_TRAINING_DATA_HASH
)


print(
    "\nThis exact dataframe will be reused "
    "for seeds 42, 43, and 44."
)


print(
    "\nFINAL UNFILTERED 1x AUGMENTED "
    "TRAINING DATA VERIFIED."
)

UNFILTERED 1x AUGMENTED TRAINING DATA VERIFICATION
Original RUHSOLD training samples: 6408
Unfiltered synthetic samples: 1448
Final augmented training samples: 7856

Synthetic class distribution:


,label,count
0,2,500
1,3,537
2,4,411



Final augmented class distribution:


,label,count
0,0,1537
1,1,3423
2,2,1000
3,3,1074
4,4,822



Fixed augmented training-data SHA256:
e3a56bdbb7d79ae0bf34d0fd9566c1f4d016bdcc2866d6ee07362043c5116049

This exact dataframe will be reused for seeds 42, 43, and 44.

FINAL UNFILTERED 1x AUGMENTED TRAINING DATA VERIFIED.


In [48]:
# ============================================================
# TOKENIZER
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__
)

Tokenizer: XLMRobertaTokenizer


In [49]:
# ============================================================
# PYTORCH DATASET CLASS
# ============================================================

from torch.utils.data import Dataset


class RUHSOLDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length
    ):

        self.tweets = (
            dataframe[
                "tweet"
            ]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe[
                "label"
            ]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(
            self.tweets
        )


    def __getitem__(
        self,
        index
    ):

        tweet = (
            self.tweets[
                index
            ]
        )

        label = (
            self.labels[
                index
            ]
        )

        encoded = (
            self.tokenizer(
                tweet,
                truncation=True,
                max_length=(
                    self.max_length
                ),
                padding=False,
            )
        )

        encoded[
            "labels"
        ] = label

        return encoded


print(
    "RUHSOLDDataset loaded."
)

RUHSOLDDataset loaded.


In [50]:
# ============================================================
# CREATE DATASET OBJECTS
#
# IMPORTANT:
# - Training dataset = original RUHSOLD + fixed unfiltered 1x
# - Validation dataset = original RUHSOLD validation only
# - Test set is NOT instantiated here
# - The same training dataset object is reused for all seeds
# ============================================================


# ============================================================
# UNFILTERED AUGMENTED TRAINING DATASET
# ============================================================

train_unfiltered_aug_dataset = (
    RUHSOLDDataset(
        dataframe=(
            train_unfiltered_aug_df
        ),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
)


# ============================================================
# ORIGINAL VALIDATION DATASET
# ============================================================

val_dataset = (
    RUHSOLDDataset(
        dataframe=val_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
)


# ============================================================
# VERIFY DATASET SIZES
# ============================================================

assert (
    len(
        train_unfiltered_aug_dataset
    )
    ==
    FINAL_UNFILTERED_TRAIN_SIZE
)


assert (
    len(
        val_dataset
    )
    ==
    801
)


# ============================================================
# VERIFY LABEL DISTRIBUTION INSIDE AUGMENTED DATASET
# ============================================================

train_unfiltered_dataset_counts = {
    int(label): int(count)

    for label, count

    in zip(
        *np.unique(
            np.asarray(
                train_unfiltered_aug_dataset.labels,
                dtype=int,
            ),
            return_counts=True,
        )
    )
}


assert (
    train_unfiltered_dataset_counts
    ==
    EXPECTED_UNFILTERED_AUGMENTED_COUNTS
)


# ============================================================
# VERIFY VALIDATION DISTRIBUTION
# ============================================================

val_dataset_counts = {
    int(label): int(count)

    for label, count

    in zip(
        *np.unique(
            np.asarray(
                val_dataset.labels,
                dtype=int,
            ),
            return_counts=True,
        )
    )
}


EXPECTED_VAL_COUNTS = {
    0: 192,
    1: 428,
    2: 63,
    3: 67,
    4: 51,
}


assert (
    val_dataset_counts
    ==
    EXPECTED_VAL_COUNTS
)


# ============================================================
# VERIFY DATAFRAME HASH STILL MATCHES
#
# This confirms the frozen augmented dataframe has not been
# changed before conversion into the Dataset object.
# ============================================================

current_training_hash = (
    dataframe_sha256(
        train_unfiltered_aug_df
    )
)


assert (
    current_training_hash
    ==
    UNFILTERED_TRAINING_DATA_HASH
)


# ============================================================
# DISPLAY VERIFICATION
# ============================================================

print("=" * 70)
print(
    "UNFILTERED DATASET OBJECT VERIFICATION"
)
print("=" * 70)


print(
    "Unfiltered augmented train:",
    len(
        train_unfiltered_aug_dataset
    )
)


print(
    "Validation:",
    len(
        val_dataset
    )
)


print(
    "\nTraining class distribution:"
)

print(
    train_unfiltered_dataset_counts
)


print(
    "Validation class distribution:"
)

print(
    val_dataset_counts
)


print(
    "\nTraining-data SHA256:"
)

print(
    current_training_hash
)


print(
    "\nTEST SET STATUS:"
)

print(
    "Not instantiated here. "
    "Reserved for final evaluation only."
)


print(
    "\nUNFILTERED TRAIN / VALIDATION "
    "DATASET OBJECTS VERIFIED."
)

UNFILTERED DATASET OBJECT VERIFICATION
Unfiltered augmented train: 7856
Validation: 801

Training class distribution:
{0: 1537, 1: 3423, 2: 1000, 3: 1074, 4: 822}
Validation class distribution:
{0: 192, 1: 428, 2: 63, 3: 67, 4: 51}

Training-data SHA256:
e3a56bdbb7d79ae0bf34d0fd9566c1f4d016bdcc2866d6ee07362043c5116049

TEST SET STATUS:
Not instantiated here. Reserved for final evaluation only.

UNFILTERED TRAIN / VALIDATION DATASET OBJECTS VERIFIED.


In [51]:
# ============================================================
# DYNAMIC BATCH PADDING
# ============================================================

from transformers import (
    DataCollatorWithPadding
)

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt"
    )
)

print(
    "Data collator loaded."
)

Data collator loaded.


In [52]:
# ============================================================
# EVALUATION METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)


def compute_metrics(
    eval_prediction
):

    logits, labels = (
        eval_prediction
    )

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = (
        accuracy_score(
            labels,
            predictions
        )
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "weighted_precision":
            weighted_precision,

        "weighted_recall":
            weighted_recall,

        "weighted_f1":
            weighted_f1,
    }


print(
    "Metric function loaded."
)

Metric function loaded.


In [53]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = (
        True
    )

    torch.backends.cudnn.benchmark = (
        False
    )


set_seed(
    42
)

print(
    "Reproducibility setup loaded."
)

Reproducibility setup loaded.


In [54]:
print("Train:", len(train_unfiltered_aug_dataset))
print("Validation:", len(val_dataset))
print("Seeds:", SEEDS)

Train: 7856
Validation: 801
Seeds: [42, 43, 44]


In [55]:
# ============================================================
# UNFILTERED 1x AUGMENTED XLM-R EXPERIMENT
# CANONICAL MATCHED CONFIGURATION
#
# IMPORTANT:
# This configuration is matched to the FINAL ORIGINAL
# RUHSOLD XLM-R baseline.
#
# Across the original and augmented XLM-R experiments:
# - same base model
# - same tokenizer / max length
# - same frozen Optuna hyperparameters
# - same physical/effective batch configuration
# - same mixed-precision strategy
# - same seeds
# - same validation set
# - same checkpoint-selection criterion
#
# The intended experimental difference is TRAINING DATA.
# ============================================================

from pathlib import Path

import os
import gc
import math
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

CLASSIFIER_DIR = (
    PROJECT_ROOT
    / "classifier"
)

FINE_TUNING_DIR = (
    PROJECT_ROOT
    / "fine tuning"
)

GENERATION_DIR = (
    FINE_TUNING_DIR
    / "outputs"
    / "synthetic_generation"
)


# ============================================================
# ORIGINAL RUHSOLD DATA PATHS
# ============================================================

TRAIN_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_train.tsv"
)

VAL_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_validation.tsv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


# ============================================================
# RAW SYNTHETIC GENERATION PATHS
# ============================================================

ROUND1_RAW_DIR = (
    GENERATION_DIR
    / "full"
)

ROUND2_RAW_DIR = (
    GENERATION_DIR
    / "full_round2"
)


# ============================================================
# FROZEN UNFILTERED SYNTHETIC DATA
# ============================================================

UNFILTERED_DATA_DIR = (
    GENERATION_DIR
    / "unfiltered_control"
)

UNFILTERED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

UNFILTERED_1X_PATH = (
    UNFILTERED_DATA_DIR
    / "unfiltered_1x_synthetic_training_set.csv"
)


# ============================================================
# UNFILTERED XLM-R OUTPUT DIRECTORIES
# ============================================================

UNFILTERED_OUTPUT_DIR = (
    CLASSIFIER_DIR
    / "outputs"
    / "xlm_roberta_unfiltered_1x_corrected"
)

UNFILTERED_RESULTS_DIR = (
    CLASSIFIER_DIR
    / "outputs"
    / "xlm_roberta_unfiltered_1x_corrected_results"
)


UNFILTERED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

UNFILTERED_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_NAME = (
    "FacebookAI/xlm-roberta-base"
)

NUM_LABELS = 5

MAX_LENGTH = 128


# ============================================================
# FROZEN OPTUNA-SELECTED HYPERPARAMETERS
#
# EXACTLY THE SAME VALUES AS THE FINAL ORIGINAL BASELINE.
# No retuning is performed.
# ============================================================

BEST_LEARNING_RATE = (
    2.881129057462248e-05
)

BEST_WEIGHT_DECAY = (
    0.03348739395854274
)

BEST_WARMUP_RATIO = (
    0.03756172525242821
)

FINAL_MAX_EPOCHS = 10

FINAL_EARLY_STOPPING_PATIENCE = 2


# ============================================================
# CANONICAL BATCH CONFIGURATION
#
# EXACT MATCH TO FINAL ORIGINAL XLM-R BASELINE:
#
# physical train batch = 16
# gradient accumulation = 1
# effective train batch = 16
# evaluation batch      = 16
# ============================================================

TRAIN_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 1

EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

EVAL_BATCH_SIZE = 16


assert (
    EFFECTIVE_BATCH_SIZE
    ==
    16
)


# Backward-compatible variable name if later notebook cells
# still refer to BEST_BATCH_SIZE.
BEST_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
)


# ============================================================
# MATCHED UNFILTERED 1x AUGMENTATION TARGET
#
# SAME FROZEN SYNTHETIC SET FOR ALL THREE TRAINING SEEDS.
# ============================================================

UNFILTERED_TARGET_COUNTS = {
    2: 500,   # Religious Hate
    3: 537,   # Sexism
    4: 411,   # Profane
}

UNFILTERED_SELECTION_SEED = 42


# ============================================================
# FINAL TRAINING SIZE
# ============================================================

ORIGINAL_TRAIN_SIZE = 6408

SYNTHETIC_1X_SIZE = 1448

FINAL_UNFILTERED_TRAIN_SIZE = (
    ORIGINAL_TRAIN_SIZE
    +
    SYNTHETIC_1X_SIZE
)


assert (
    FINAL_UNFILTERED_TRAIN_SIZE
    ==
    7856
)


# ============================================================
# TRAINING STEP CALCULATION
#
# IMPORTANT:
# The frozen hyperparameter is the WARMUP RATIO.
#
# Because this augmented dataset contains 7,856 examples,
# the same ratio corresponds to a different absolute number
# of warmup steps than the 6,408-example original baseline.
#
# 7856 / 16
# = 491 mini-batches per epoch
#
# gradient accumulation = 1
#
# optimizer steps per epoch
# = 491
#
# 491 x 10
# = 4910 maximum optimizer steps
#
# ceil(
#     4910 x 0.03756172525242821
# )
# = 185 warmup steps
# ============================================================

MINI_BATCHES_PER_EPOCH = math.ceil(
    FINAL_UNFILTERED_TRAIN_SIZE
    / TRAIN_BATCH_SIZE
)


OPTIMIZER_STEPS_PER_EPOCH = math.ceil(
    MINI_BATCHES_PER_EPOCH
    / GRADIENT_ACCUMULATION_STEPS
)


MAX_OPTIMIZER_STEPS = (
    OPTIMIZER_STEPS_PER_EPOCH
    * FINAL_MAX_EPOCHS
)


BEST_WARMUP_STEPS = math.ceil(
    MAX_OPTIMIZER_STEPS
    * BEST_WARMUP_RATIO
)


assert (
    MINI_BATCHES_PER_EPOCH
    ==
    491
)

assert (
    OPTIMIZER_STEPS_PER_EPOCH
    ==
    491
)

assert (
    MAX_OPTIMIZER_STEPS
    ==
    4910
)

assert (
    BEST_WARMUP_STEPS
    ==
    185
)


# ============================================================
# REPEATED-RUN SEEDS
# ============================================================

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: class_id
    for class_id, label
    in id2label.items()
}


# ============================================================
# EXPECTED FINAL CLASS DISTRIBUTION
# ============================================================

EXPECTED_UNFILTERED_AUGMENTED_COUNTS = {
    0: 1537,
    1: 3423,
    2: 1000,
    3: 1074,
    4: 822,
}


# ============================================================
# BASE REPRODUCIBILITY SETUP
# ============================================================

GLOBAL_SEED = 42


random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)

torch.manual_seed(
    GLOBAL_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        GLOBAL_SEED
    )


# ============================================================
# DISPLAY + VERIFY EXPERIMENT CONFIGURATION
# ============================================================

print("=" * 70)

print(
    "UNFILTERED 1x AUGMENTED XLM-R "
    "CANONICAL MATCHED EXPERIMENT"
)

print("=" * 70)


print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print(
    "\nModel:",
    MODEL_NAME
)


print(
    "\nOriginal training samples:",
    ORIGINAL_TRAIN_SIZE
)

print(
    "Frozen unfiltered synthetic samples:",
    SYNTHETIC_1X_SIZE
)

print(
    "Final training samples:",
    FINAL_UNFILTERED_TRAIN_SIZE
)


print(
    "\nFrozen hyperparameters:"
)

print(
    "Learning rate:",
    BEST_LEARNING_RATE
)

print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)

print(
    "Frozen warmup ratio:",
    BEST_WARMUP_RATIO
)


print(
    "\nCanonical batch configuration:"
)

print(
    "Physical training batch size:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation steps:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective training batch size:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch size:",
    EVAL_BATCH_SIZE
)


print(
    "\nTraining schedule:"
)

print(
    "Mini-batches per epoch:",
    MINI_BATCHES_PER_EPOCH
)

print(
    "Optimizer steps per epoch:",
    OPTIMIZER_STEPS_PER_EPOCH
)

print(
    "Maximum optimizer steps:",
    MAX_OPTIMIZER_STEPS
)

print(
    "Equivalent warmup steps:",
    BEST_WARMUP_STEPS
)


print(
    "\nMaximum epochs:",
    FINAL_MAX_EPOCHS
)

print(
    "Early stopping patience:",
    FINAL_EARLY_STOPPING_PATIENCE
)

print(
    "Seeds:",
    SEEDS
)


print(
    "\nOutput directory:"
)

print(
    UNFILTERED_OUTPUT_DIR
)


print(
    "\nResults directory:"
)

print(
    UNFILTERED_RESULTS_DIR
)


print(
    "\nFrozen unfiltered synthetic file:"
)

print(
    UNFILTERED_1X_PATH
)


print(
    "\nCANONICAL UNFILTERED CONFIGURATION VERIFIED."
)

UNFILTERED 1x AUGMENTED XLM-R CANONICAL MATCHED EXPERIMENT
CUDA available: True
GPU: NVIDIA L40S

Model: FacebookAI/xlm-roberta-base

Original training samples: 6408
Frozen unfiltered synthetic samples: 1448
Final training samples: 7856

Frozen hyperparameters:
Learning rate: 2.881129057462248e-05
Weight decay: 0.03348739395854274
Frozen warmup ratio: 0.03756172525242821

Canonical batch configuration:
Physical training batch size: 16
Gradient accumulation steps: 1
Effective training batch size: 16
Evaluation batch size: 16

Training schedule:
Mini-batches per epoch: 491
Optimizer steps per epoch: 491
Maximum optimizer steps: 4910
Equivalent warmup steps: 185

Maximum epochs: 10
Early stopping patience: 2
Seeds: [42, 43, 44]

Output directory:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected

Results directory:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected_results

Frozen unfiltered synthet

In [56]:
# ============================================================
# CLEAN FAILED TRAINING STATE
# ============================================================

import gc
import torch

for name in [
    "trainer",
    "model",
    "training_args",
]:

    if name in globals():

        try:
            del globals()[name]
        except:
            pass


gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


free, total = torch.cuda.mem_get_info()

print(
    f"CUDA free : {free / 1024**3:.2f} GB"
)

print(
    f"CUDA total: {total / 1024**3:.2f} GB"
)

CUDA free : 15.58 GB
CUDA total: 16.00 GB


In [57]:
# ============================================================
# UNFILTERED 1x AUGMENTED XLM-R
# CORRECTED THREE-SEED VALIDATION + PER-CLASS EXPERIMENT
#
# ISSUE-1 FIX:
# - XLM-R is loaded normally in FP32
# - NO dtype=torch.bfloat16 in from_pretrained()
# - BF16 is used ONLY through TrainingArguments
#
# CANONICAL MATCH:
# - train batch = 16
# - gradient accumulation = 1
# - effective batch = 16
# - eval batch = 16
# - same frozen Optuna hyperparameters
# - same warmup ratio
# - augmented equivalent warmup steps = 185
# - same fixed synthetic dataset for all three seeds
# - best checkpoint selected by validation Macro-F1
# ============================================================

import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    classification_report,
)

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)


# ============================================================
# FINAL PRE-TRAINING AUDIT
# ============================================================

print("=" * 80)
print(
    "CORRECTED UNFILTERED 1x XLM-R - PRE-TRAINING AUDIT"
)
print("=" * 80)


print(
    "Training samples:",
    len(
        train_unfiltered_aug_dataset
    )
)

print(
    "Validation samples:",
    len(
        val_dataset
    )
)

print(
    "Seeds:",
    SEEDS
)


print(
    "\nFrozen hyperparameters:"
)

print(
    "Learning rate:",
    BEST_LEARNING_RATE
)

print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)

print(
    "Warmup ratio:",
    BEST_WARMUP_RATIO
)

print(
    "Warmup steps:",
    BEST_WARMUP_STEPS
)


print(
    "\nCanonical batch configuration:"
)

print(
    "Physical train batch:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)


print(
    "\nFixed training-data hash:"
)

print(
    UNFILTERED_TRAINING_DATA_HASH
)


# ============================================================
# REQUIRED ASSERTIONS
# ============================================================

assert (
    len(
        train_unfiltered_aug_dataset
    )
    ==
    7856
)


assert (
    len(
        val_dataset
    )
    ==
    801
)


assert (
    TRAIN_BATCH_SIZE
    ==
    16
)


assert (
    GRADIENT_ACCUMULATION_STEPS
    ==
    1
)


assert (
    EFFECTIVE_BATCH_SIZE
    ==
    16
)


assert (
    EVAL_BATCH_SIZE
    ==
    16
)


assert (
    BEST_WARMUP_STEPS
    ==
    185
)


assert (
    SEEDS
    ==
    [
        42,
        43,
        44,
    ]
)


print(
    "\nPre-training configuration audit: PASSED"
)


# ============================================================
# RESULT STORAGE
# ============================================================

unfiltered_overall_results = []

unfiltered_per_class_results = []


# ============================================================
# THREE-SEED EXPERIMENT
# ============================================================

for seed in SEEDS:

    print("\n")
    print("=" * 80)

    print(
        f"CORRECTED UNFILTERED 1x XLM-R - SEED {seed}"
    )

    print("=" * 80)


    # ========================================================
    # VERIFY FIXED DATASET HAS NOT CHANGED
    # ========================================================

    current_training_hash = (
        dataframe_sha256(
            train_unfiltered_aug_df
        )
    )


    assert (
        current_training_hash
        ==
        UNFILTERED_TRAINING_DATA_HASH
    ), (
        f"Training dataset changed before seed {seed}."
    )


    print(
        "Training-data hash:",
        current_training_hash
    )

    print(
        "Fixed dataset integrity: PASSED"
    )


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    set_seed(
        seed
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # SEED-SPECIFIC OUTPUT DIRECTORY
    #
    # Note:
    # UNFILTERED_OUTPUT_DIR should point to the new
    # xlm_roberta_unfiltered_1x_corrected directory.
    # ========================================================

    seed_output_dir = (
        UNFILTERED_OUTPUT_DIR
        / f"seed_{seed}"
    )


    # --------------------------------------------------------
    # Remove only an incomplete/old corrected run for this seed.
    # This does NOT touch the old direct-BF16 experiment.
    # --------------------------------------------------------

    if seed_output_dir.exists():

        print(
            "Removing old corrected seed directory:"
        )

        print(
            seed_output_dir
        )

        shutil.rmtree(
            seed_output_dir
        )


    seed_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # LOAD FRESH PRETRAINED XLM-R
    #
    # CRITICAL ISSUE-1 FIX:
    #
    # DO NOT use:
    #
    # dtype=torch.bfloat16
    #
    # Model parameters must initially load as FP32.
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,

            num_labels=NUM_LABELS,

            id2label=id2label,

            label2id=label2id,
        )
    )


    model_parameter_dtype = (
        next(
            model.parameters()
        ).dtype
    )


    print(
        "\nModel parameter dtype before Trainer:",
        model_parameter_dtype
    )


    assert (
        model_parameter_dtype
        ==
        torch.float32
    ), (
        f"ERROR: Model loaded as {model_parameter_dtype}. "
        "Expected torch.float32."
    )


    print(
        "FP32 pretrained model loading: VERIFIED"
    )


    # ========================================================
    # TRAINING ARGUMENTS
    # ========================================================

    training_args = TrainingArguments(

        output_dir=str(
            seed_output_dir
        ),

        # ----------------------------------------------------
        # Evaluation / logging
        # ----------------------------------------------------

        eval_strategy="epoch",

        logging_strategy="epoch",

        # ----------------------------------------------------
        # Save validation-selected best checkpoint only
        # ----------------------------------------------------

        save_strategy="best",

        save_total_limit=1,

        save_only_model=True,

        # ----------------------------------------------------
        # Frozen Optuna hyperparameters
        # ----------------------------------------------------

        learning_rate=(
            BEST_LEARNING_RATE
        ),

        weight_decay=(
            BEST_WEIGHT_DECAY
        ),

        # ----------------------------------------------------
        # Canonical matched batch configuration
        # ----------------------------------------------------

        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        # ----------------------------------------------------
        # Same frozen warmup ratio
        #
        # 7,856-sample equivalent = 185 optimizer steps
        # ----------------------------------------------------

        warmup_steps=(
            BEST_WARMUP_STEPS
        ),

        # ----------------------------------------------------
        # Maximum training duration
        # ----------------------------------------------------

        num_train_epochs=(
            FINAL_MAX_EPOCHS
        ),

        # ----------------------------------------------------
        # Validation-only checkpoint selection
        # ----------------------------------------------------

        load_best_model_at_end=True,

        metric_for_best_model=(
            CHECKPOINT_SELECTION_METRIC
        ),

        greater_is_better=True,

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------

        seed=seed,

        data_seed=seed,

        # ----------------------------------------------------
        # Mixed precision
        #
        # IMPORTANT:
        # Model was loaded in FP32.
        # BF16 is used only for Trainer mixed precision.
        # ----------------------------------------------------

        bf16=(
            torch.cuda.is_available()
        ),

        fp16=False,

        # ----------------------------------------------------
        # Misc
        # ----------------------------------------------------

        report_to="none",

        disable_tqdm=False,
    )


    # ========================================================
    # TRAINING ARGUMENT AUDIT
    # ========================================================

    print(
        "\nTraining configuration:"
    )

    print(
        "Physical train batch:",
        training_args.per_device_train_batch_size
    )

    print(
        "Gradient accumulation:",
        training_args.gradient_accumulation_steps
    )

    print(
        "Effective batch:",
        (
            training_args.per_device_train_batch_size
            *
            training_args.gradient_accumulation_steps
        )
    )

    print(
        "Evaluation batch:",
        training_args.per_device_eval_batch_size
    )

    print(
        "Warmup steps:",
        training_args.warmup_steps
    )

    print(
        "Trainer BF16:",
        training_args.bf16
    )


    assert (
        training_args.per_device_train_batch_size
        ==
        16
    )

    assert (
        training_args.gradient_accumulation_steps
        ==
        1
    )

    assert (
        training_args.per_device_eval_batch_size
        ==
        16
    )

    assert (
        training_args.warmup_steps
        ==
        185
    )


    # ========================================================
    # TRAINER
    #
    # CRITICAL:
    # - training = fixed unfiltered augmented dataset
    # - validation = unchanged original RUHSOLD validation
    # ========================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=(
            train_unfiltered_aug_dataset
        ),

        eval_dataset=(
            val_dataset
        ),

        data_collator=(
            data_collator
        ),

        compute_metrics=(
            compute_metrics
        ),

        processing_class=(
            tokenizer
        ),

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=(
                    FINAL_EARLY_STOPPING_PATIENCE
                ),

                early_stopping_threshold=0.0,
            )
        ],
    )


    # ========================================================
    # TRAIN
    # ========================================================

    print(
        "\nStarting corrected training..."
    )


    start_time = time.time()


    trainer.train()


    training_time = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # BEST CHECKPOINT INFORMATION
    # ========================================================

    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )


    best_validation_macro_f1 = (
        trainer.state.best_metric
    )


    epoch_reached = (
        trainer.state.epoch
    )


    assert (
        best_checkpoint
        is not None
    )


    best_checkpoint_path = Path(
        best_checkpoint
    )


    assert (
        best_checkpoint_path.exists()
    )


    print(
        "\nBest checkpoint:"
    )

    print(
        best_checkpoint
    )


    print(
        "Best validation Macro F1:",
        f"{best_validation_macro_f1:.4f}"
    )


    print(
        "Epoch reached:",
        epoch_reached
    )


    print(
        "Training time:",
        f"{training_time / 60:.2f} minutes"
    )


    # ========================================================
    # VALIDATION PREDICTIONS FROM BEST CHECKPOINT
    #
    # load_best_model_at_end=True means Trainer has restored
    # the validation-selected best checkpoint.
    # ========================================================

    prediction_output = (
        trainer.predict(
            val_dataset
        )
    )


    y_true = (
        prediction_output.label_ids
    )


    y_pred = np.argmax(
        prediction_output.predictions,
        axis=1,
    )


    assert (
        len(
            y_true
        )
        ==
        801
    )


    assert (
        len(
            y_pred
        )
        ==
        801
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_true,

        y_pred,

        labels=(
            METRIC_LABELS
        ),

        target_names=[
            id2label[
                class_id
            ]
            for class_id
            in METRIC_LABELS
        ],

        output_dict=True,

        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    print(
        f"\nPER-CLASS VALIDATION RESULTS - SEED {seed}"
    )


    display(
        report_df.round(4)
    )


    # ========================================================
    # STORE OVERALL METRICS
    # ========================================================

    overall_result = {

        "seed":
            seed,

        "best_checkpoint":
            str(
                best_checkpoint
            ),

        "epoch_reached":
            epoch_reached,

        "best_validation_macro_f1":
            best_validation_macro_f1,

        "validation_accuracy":
            report[
                "accuracy"
            ],

        "validation_macro_precision":
            report[
                "macro avg"
            ][
                "precision"
            ],

        "validation_macro_recall":
            report[
                "macro avg"
            ][
                "recall"
            ],

        "validation_macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ],

        "validation_weighted_precision":
            report[
                "weighted avg"
            ][
                "precision"
            ],

        "validation_weighted_recall":
            report[
                "weighted avg"
            ][
                "recall"
            ],

        "validation_weighted_f1":
            report[
                "weighted avg"
            ][
                "f1-score"
            ],

        "training_time_minutes":
            training_time / 60,

        "training_data_sha256":
            current_training_hash,

        "model_initial_dtype":
            str(
                model_parameter_dtype
            ),

        "physical_train_batch":
            TRAIN_BATCH_SIZE,

        "gradient_accumulation":
            GRADIENT_ACCUMULATION_STEPS,

        "effective_batch":
            EFFECTIVE_BATCH_SIZE,

        "eval_batch":
            EVAL_BATCH_SIZE,

        "warmup_steps":
            BEST_WARMUP_STEPS,
    }


    unfiltered_overall_results.append(
        overall_result
    )


    # ========================================================
    # STORE PER-CLASS METRICS
    # ========================================================

    seed_per_class_rows = []


    for class_id in (
        METRIC_LABELS
    ):

        class_name = (
            id2label[
                class_id
            ]
        )


        class_metrics = (
            report[
                class_name
            ]
        )


        row = {

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        }


        seed_per_class_rows.append(
            row
        )


        unfiltered_per_class_results.append(
            row
        )


    seed_per_class_df = (
        pd.DataFrame(
            seed_per_class_rows
        )
    )


    # ========================================================
    # SAVE VALIDATION PREDICTIONS
    # ========================================================

    prediction_df = pd.DataFrame({

        "true_label_id":
            y_true,

        "predicted_label_id":
            y_pred,

        "true_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_true
            ],

        "predicted_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_pred
            ],
    })


    prediction_df.to_csv(

        UNFILTERED_RESULTS_DIR
        / (
            f"seed_{seed}_"
            "validation_predictions.csv"
        ),

        index=False,
    )


    # ========================================================
    # SAVE CUMULATIVE RESULTS AFTER EACH SEED
    #
    # Important for server interruptions.
    # ========================================================

    pd.DataFrame(
        unfiltered_overall_results
    ).to_csv(

        UNFILTERED_RESULTS_DIR
        / "unfiltered_1x_corrected_3seed_overall_results.csv",

        index=False,
    )


    pd.DataFrame(
        unfiltered_per_class_results
    ).to_csv(

        UNFILTERED_RESULTS_DIR
        / "unfiltered_1x_corrected_3seed_per_class_results.csv",

        index=False,
    )


    # ========================================================
    # RETAIN BEST CHECKPOINT
    # ========================================================

    assert (
        best_checkpoint_path.exists()
    )


    print(
        "\nKeeping best checkpoint for final test evaluation:"
    )

    print(
        best_checkpoint_path
    )


    # ========================================================
    # MEMORY CLEANUP ONLY
    #
    # DO NOT delete checkpoint from disk.
    # ========================================================

    del prediction_output
    del trainer
    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Verify checkpoint survived cleanup
    # --------------------------------------------------------

    assert (
        best_checkpoint_path.exists()
    )


    print(
        "Checkpoint confirmed after memory cleanup:"
    )

    print(
        best_checkpoint_path
    )


    # --------------------------------------------------------
    # Disk-space report
    # --------------------------------------------------------

    total, used, free = (
        shutil.disk_usage(
            "/home/jovyan"
        )
    )


    print(
        "Disk free after cleanup:",
        f"{free / 1024**3:.2f} GB"
    )


# ============================================================
# FINAL THREE-SEED OVERALL RESULTS
# ============================================================

unfiltered_overall_results_df = (
    pd.DataFrame(
        unfiltered_overall_results
    )
)


assert (
    len(
        unfiltered_overall_results_df
    )
    ==
    3
)


print("\n")
print("=" * 80)

print(
    "CORRECTED UNFILTERED 1x AUGMENTED "
    "THREE-SEED EXPERIMENT COMPLETE"
)

print("=" * 80)


print(
    "\nPer-seed overall results:"
)


display(
    unfiltered_overall_results_df.round(4)
)


# ============================================================
# OVERALL MEAN ± SAMPLE STANDARD DEVIATION
# ============================================================

summary_columns = [

    "validation_accuracy",

    "validation_macro_precision",

    "validation_macro_recall",

    "validation_macro_f1",

    "validation_weighted_f1",
]


unfiltered_overall_summary_df = (

    unfiltered_overall_results_df[
        summary_columns
    ]

    .agg(
        [
            "mean",
            "std",
        ]
    )
)


print(
    "\nThree-seed overall validation summary:"
)


display(
    unfiltered_overall_summary_df.round(4)
)


print(
    "\nMean validation Macro F1:",
    f'{unfiltered_overall_results_df["validation_macro_f1"].mean():.4f}'
)


print(
    "Validation Macro F1 standard deviation:",
    f'{unfiltered_overall_results_df["validation_macro_f1"].std(ddof=1):.4f}'
)


# ============================================================
# THREE-SEED PER-CLASS SUMMARY
# ============================================================

unfiltered_per_class_results_df = (
    pd.DataFrame(
        unfiltered_per_class_results
    )
)


assert (
    len(
        unfiltered_per_class_results_df
    )
    ==
    15
)


unfiltered_per_class_summary_df = (

    unfiltered_per_class_results_df

    .groupby(
        [
            "class_id",
            "class_name",
        ]
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )

    .reset_index()
)


print(
    "\nThree-seed per-class validation summary:"
)


display(
    unfiltered_per_class_summary_df.round(4)
)


# ============================================================
# SAVE FINAL SUMMARIES
# ============================================================

unfiltered_overall_summary_df.to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_overall_summary.csv"
)


unfiltered_per_class_summary_df.to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_per_class_summary.csv",

    index=False,
)


# ============================================================
# FINAL CHECKPOINT VERIFICATION
# ============================================================

print("\n")
print("=" * 80)

print(
    "RETAINED CORRECTED UNFILTERED CHECKPOINTS"
)

print("=" * 80)


FINAL_UNFILTERED_CHECKPOINTS = {}


for seed in SEEDS:

    seed_dir = (
        UNFILTERED_OUTPUT_DIR
        / f"seed_{seed}"
    )


    checkpoints = sorted(
        seed_dir.glob(
            "checkpoint-*"
        )
    )


    assert (
        len(
            checkpoints
        )
        ==
        1
    ), (
        f"Expected exactly one retained checkpoint "
        f"for seed {seed}; found {len(checkpoints)}."
    )


    FINAL_UNFILTERED_CHECKPOINTS[
        seed
    ] = checkpoints[0]


    print(
        f"\nSeed {seed}:"
    )

    print(
        checkpoints[0]
    )


# ============================================================
# FINAL DATASET-HASH CONSISTENCY CHECK
# ============================================================

assert (
    unfiltered_overall_results_df[
        "training_data_sha256"
    ]
    .nunique()
    ==
    1
)


assert (
    unfiltered_overall_results_df[
        "training_data_sha256"
    ]
    .iloc[0]
    ==
    UNFILTERED_TRAINING_DATA_HASH
)


print(
    "\nCross-seed training-data hash consistency: PASSED"
)


print(
    "\nAll corrected unfiltered results saved to:"
)

print(
    UNFILTERED_RESULTS_DIR
)

CORRECTED UNFILTERED 1x XLM-R - PRE-TRAINING AUDIT
Training samples: 7856
Validation samples: 801
Seeds: [42, 43, 44]

Frozen hyperparameters:
Learning rate: 2.881129057462248e-05
Weight decay: 0.03348739395854274
Warmup ratio: 0.03756172525242821
Warmup steps: 185

Canonical batch configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16

Fixed training-data hash:
e3a56bdbb7d79ae0bf34d0fd9566c1f4d016bdcc2866d6ee07362043c5116049

Pre-training configuration audit: PASSED


CORRECTED UNFILTERED 1x XLM-R - SEED 42
Training-data hash: e3a56bdbb7d79ae0bf34d0fd9566c1f4d016bdcc2866d6ee07362043c5116049
Fixed dataset integrity: PASSED
Removing old corrected seed directory:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[HAMI-core Msg(1203:139941170345792:multiprocess_memory_limit.c:455)]: Calling exit


Model parameter dtype before Trainer: torch.float32
FP32 pretrained model loading: VERIFIED

Training configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16
Warmup steps: 185
Trainer BF16: True

Starting corrected training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,1.260930,0.758969,0.725343,0.633045,0.667037,0.644911,0.725161,0.725343,0.721812
2,0.895016,0.755732,0.739076,0.643626,0.700364,0.652191,0.754129,0.739076,0.732783
3,0.763635,0.736418,0.752809,0.657015,0.728337,0.667683,0.775763,0.752809,0.747143
4,0.652634,0.803947,0.725343,0.625926,0.712826,0.658142,0.757773,0.725343,0.734061
5,0.540815,0.687258,0.777778,0.683778,0.716482,0.688984,0.788691,0.777778,0.775791
6,0.450359,0.871627,0.739076,0.630073,0.726048,0.648334,0.779737,0.739076,0.739782
7,0.370533,0.821453,0.766542,0.667250,0.735407,0.691445,0.784354,0.766542,0.770106
8,0.300223,0.937014,0.752809,0.647140,0.742431,0.677650,0.785756,0.752809,0.759889
9,0.248608,0.919358,0.779026,0.676072,0.744303,0.701696,0.794568,0.779026,0.782153
10,0.201996,1.009510,0.762797,0.655449,0.736726,0.682097,0.787777,0.762797,0.767308


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best checkpoint:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42/checkpoint-4419
Best validation Macro F1: 0.7017
Epoch reached: 10.0
Training time: 5.36 minutes


NameError: name 'METRIC_LABELS' is not defined

In [58]:
# ============================================================
# FIXED METRIC LABEL CONFIGURATION
# ============================================================

METRIC_LABELS = [
    0,
    1,
    2,
    3,
    4,
]

CHECKPOINT_SELECTION_METRIC = "macro_f1"

print("Metric labels:", METRIC_LABELS)
print("Checkpoint selection metric:", CHECKPOINT_SELECTION_METRIC)

Metric labels: [0, 1, 2, 3, 4]
Checkpoint selection metric: macro_f1


In [59]:
# ============================================================
# RECOVER COMPLETED SEED 42 AFTER METRIC_LABELS ERROR
# ============================================================

seed = 42

print("=" * 80)
print("RECOVERING COMPLETED UNFILTERED SEED 42")
print("=" * 80)


# ============================================================
# CLASSIFICATION REPORT
# y_true and y_pred already exist from the completed prediction
# ============================================================

report = classification_report(
    y_true,
    y_pred,

    labels=METRIC_LABELS,

    target_names=[
        id2label[class_id]
        for class_id
        in METRIC_LABELS
    ],

    output_dict=True,
    zero_division=0,
)


report_df = (
    pd.DataFrame(report)
    .transpose()
)


print(
    "\nPER-CLASS VALIDATION RESULTS - SEED 42"
)

display(
    report_df.round(4)
)


# ============================================================
# STORE OVERALL RESULT
# ============================================================

overall_result = {

    "seed":
        seed,

    "best_checkpoint":
        str(best_checkpoint),

    "epoch_reached":
        epoch_reached,

    "best_validation_macro_f1":
        best_validation_macro_f1,

    "validation_accuracy":
        report["accuracy"],

    "validation_macro_precision":
        report["macro avg"]["precision"],

    "validation_macro_recall":
        report["macro avg"]["recall"],

    "validation_macro_f1":
        report["macro avg"]["f1-score"],

    "validation_weighted_precision":
        report["weighted avg"]["precision"],

    "validation_weighted_recall":
        report["weighted avg"]["recall"],

    "validation_weighted_f1":
        report["weighted avg"]["f1-score"],

    "training_time_minutes":
        training_time / 60,

    "training_data_sha256":
        current_training_hash,

    "model_initial_dtype":
        str(model_parameter_dtype),

    "physical_train_batch":
        TRAIN_BATCH_SIZE,

    "gradient_accumulation":
        GRADIENT_ACCUMULATION_STEPS,

    "effective_batch":
        EFFECTIVE_BATCH_SIZE,

    "eval_batch":
        EVAL_BATCH_SIZE,

    "warmup_steps":
        BEST_WARMUP_STEPS,
}


unfiltered_overall_results.append(
    overall_result
)


# ============================================================
# STORE PER-CLASS RESULTS
# ============================================================

for class_id in METRIC_LABELS:

    class_name = id2label[class_id]

    class_metrics = report[
        class_name
    ]

    unfiltered_per_class_results.append({

        "seed":
            seed,

        "class_id":
            class_id,

        "class_name":
            class_name,

        "precision":
            class_metrics["precision"],

        "recall":
            class_metrics["recall"],

        "f1":
            class_metrics["f1-score"],

        "support":
            class_metrics["support"],
    })


# ============================================================
# SAVE VALIDATION PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "true_label_id":
        y_true,

    "predicted_label_id":
        y_pred,

    "true_label":
        [
            id2label[int(label)]
            for label
            in y_true
        ],

    "predicted_label":
        [
            id2label[int(label)]
            for label
            in y_pred
        ],
})


prediction_df.to_csv(

    UNFILTERED_RESULTS_DIR
    / "seed_42_validation_predictions.csv",

    index=False,
)


# ============================================================
# SAVE CURRENT RESULTS
# ============================================================

pd.DataFrame(
    unfiltered_overall_results
).to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_overall_results.csv",

    index=False,
)


pd.DataFrame(
    unfiltered_per_class_results
).to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_per_class_results.csv",

    index=False,
)


# ============================================================
# VERIFY CHECKPOINT
# ============================================================

best_checkpoint_path = Path(
    best_checkpoint
)

assert best_checkpoint_path.exists()


print(
    "\nSeed 42 checkpoint safely retained:"
)

print(
    best_checkpoint_path
)


# ============================================================
# MEMORY CLEANUP
# ============================================================

del prediction_output
del trainer
del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


assert best_checkpoint_path.exists()


print(
    "\nSEED 42 RECOVERY COMPLETE."
)

print(
    "Validation Macro F1:",
    f'{overall_result["validation_macro_f1"]:.4f}'
)

RECOVERING COMPLETED UNFILTERED SEED 42

PER-CLASS VALIDATION RESULTS - SEED 42


,precision,recall,f1-score,support
Abusive/Offensive,0.7564,0.6146,0.6782,192.000
Normal,0.9022,0.8621,0.8817,428.000
Religious Hate,0.6000,0.7619,0.6713,63.000
Sexism,0.6292,0.8358,0.7179,67.000
Profane,0.4925,0.6471,0.5593,51.000
accuracy,0.7790,0.7790,0.7790,0.779
macro avg,0.6761,0.7443,0.7017,801.000
weighted avg,0.7946,0.7790,0.7822,801.000



Seed 42 checkpoint safely retained:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42/checkpoint-4419

SEED 42 RECOVERY COMPLETE.
Validation Macro F1: 0.7017


In [68]:
# ============================================================
# CORRECTED UNFILTERED 1x AUGMENTED XLM-R
# FINAL CONTINUATION - SEED 44 ONLY
#
# CURRENT STATUS:
# - Seed 42 complete
# - Seed 43 complete
# - Seed 44 must be trained
#
# IMPORTANT:
# - Seeds 42 and 43 are NOT retrained
# - Same frozen augmented dataset
# - Same FP32 model loading
# - BF16 only through TrainingArguments
# - Same frozen hyperparameters
# - Best checkpoint selected by validation Macro-F1
# ============================================================

import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import classification_report

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)


# ============================================================
# REMAINING SEED ONLY
# ============================================================

REMAINING_SEEDS = [
    44,
]


# ============================================================
# KNOWN COMPLETED CHECKPOINTS
# ============================================================

SEED42_CHECKPOINT = (
    UNFILTERED_OUTPUT_DIR
    / "seed_42"
    / "checkpoint-4419"
)

SEED43_CHECKPOINT = (
    UNFILTERED_OUTPUT_DIR
    / "seed_43"
    / "checkpoint-1964"
)


assert SEED42_CHECKPOINT.exists(), (
    "Completed seed 42 checkpoint is missing."
)

assert SEED43_CHECKPOINT.exists(), (
    "Completed seed 43 checkpoint is missing."
)


print("=" * 80)
print(
    "CORRECTED UNFILTERED FINAL CONTINUATION"
)
print("=" * 80)


print(
    "Seed 42 checkpoint:"
)

print(
    SEED42_CHECKPOINT
)


print(
    "\nSeed 43 checkpoint:"
)

print(
    SEED43_CHECKPOINT
)


print(
    "\nRemaining seed:",
    REMAINING_SEEDS
)


# ============================================================
# GLOBAL CONFIGURATION VERIFICATION
# ============================================================

assert (
    len(
        train_unfiltered_aug_dataset
    )
    ==
    7856
)

assert (
    len(
        val_dataset
    )
    ==
    801
)

assert (
    TRAIN_BATCH_SIZE
    ==
    16
)

assert (
    GRADIENT_ACCUMULATION_STEPS
    ==
    1
)

assert (
    EFFECTIVE_BATCH_SIZE
    ==
    16
)

assert (
    EVAL_BATCH_SIZE
    ==
    16
)

assert (
    BEST_WARMUP_STEPS
    ==
    185
)

assert (
    CHECKPOINT_SELECTION_METRIC
    ==
    "macro_f1"
)

assert (
    METRIC_LABELS
    ==
    [
        0,
        1,
        2,
        3,
        4,
    ]
)


print(
    "\nConfiguration verification: PASSED"
)


# ============================================================
# DISK SAFETY CHECK
# ============================================================

total, used, free = shutil.disk_usage(
    "/home/jovyan"
)

free_gb = (
    free
    /
    1024**3
)


print(
    "\nFree disk space:",
    f"{free_gb:.2f} GB"
)


assert (
    free_gb
    >=
    5.0
), (
    f"Only {free_gb:.2f} GB free. "
    "Do not start seed 44."
)


print(
    "Disk-space check: PASSED"
)


# ============================================================
# LOAD EXISTING SEED 42 + 43 RESULTS
# ============================================================

overall_results_path = (
    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_overall_results.csv"
)

per_class_results_path = (
    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_per_class_results.csv"
)


assert (
    overall_results_path.exists()
), (
    "Existing overall results file is missing."
)


assert (
    per_class_results_path.exists()
), (
    "Existing per-class results file is missing."
)


existing_overall_df = pd.read_csv(
    overall_results_path
)

existing_per_class_df = pd.read_csv(
    per_class_results_path
)


unfiltered_overall_results = (
    existing_overall_df
    .to_dict(
        orient="records"
    )
)

unfiltered_per_class_results = (
    existing_per_class_df
    .to_dict(
        orient="records"
    )
)


# ============================================================
# VERIFY STORED SEEDS ARE EXACTLY 42 AND 43
# ============================================================

existing_seeds = sorted(
    {
        int(
            row["seed"]
        )
        for row
        in unfiltered_overall_results
    }
)


print(
    "\nExisting stored seeds:",
    existing_seeds
)


assert (
    existing_seeds
    ==
    [
        42,
        43,
    ]
), (
    f"Expected stored seeds [42, 43], "
    f"but found {existing_seeds}."
)


print(
    "Existing result-state verification: PASSED"
)


# ============================================================
# VERIFY EXISTING PER-CLASS ROWS
#
# 2 completed seeds x 5 classes = 10 rows
# ============================================================

assert (
    len(
        unfiltered_per_class_results
    )
    ==
    10
), (
    "Expected 10 per-class rows for completed "
    "seeds 42 and 43."
)


# ============================================================
# TRAIN SEED 44 ONLY
# ============================================================

for seed in REMAINING_SEEDS:

    print("\n")
    print("=" * 80)

    print(
        f"CORRECTED UNFILTERED 1x XLM-R - SEED {seed}"
    )

    print("=" * 80)


    # ========================================================
    # VERIFY FIXED DATASET
    # ========================================================

    current_training_hash = (
        dataframe_sha256(
            train_unfiltered_aug_df
        )
    )


    assert (
        current_training_hash
        ==
        UNFILTERED_TRAINING_DATA_HASH
    ), (
        f"Training dataset changed before seed {seed}."
    )


    print(
        "Training-data hash:",
        current_training_hash
    )

    print(
        "Fixed dataset integrity: PASSED"
    )


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    set_seed(
        seed
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    # ========================================================
    # SEED-44 OUTPUT DIRECTORY
    # ========================================================

    seed_output_dir = (
        UNFILTERED_OUTPUT_DIR
        / f"seed_{seed}"
    )


    # Remove only any failed/incomplete seed-44 attempt.
    if seed_output_dir.exists():

        print(
            "\nRemoving old/incomplete seed 44 directory:"
        )

        print(
            seed_output_dir
        )

        shutil.rmtree(
            seed_output_dir
        )


    seed_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # LOAD FRESH XLM-R IN FP32
    #
    # CRITICAL ISSUE-1 FIX:
    # NO dtype=torch.bfloat16
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,

            num_labels=NUM_LABELS,

            id2label=id2label,

            label2id=label2id,
        )
    )


    model_parameter_dtype = (
        next(
            model.parameters()
        ).dtype
    )


    print(
        "\nModel parameter dtype before Trainer:",
        model_parameter_dtype
    )


    assert (
        model_parameter_dtype
        ==
        torch.float32
    ), (
        f"Model loaded as {model_parameter_dtype}. "
        "Expected torch.float32."
    )


    print(
        "FP32 pretrained model loading: VERIFIED"
    )


    # ========================================================
    # TRAINING ARGUMENTS
    # ========================================================

    training_args = TrainingArguments(

        output_dir=str(
            seed_output_dir
        ),

        # Evaluation
        eval_strategy="epoch",

        logging_strategy="epoch",

        # Save best checkpoint only
        save_strategy="best",

        save_total_limit=1,

        save_only_model=True,

        # Frozen hyperparameters
        learning_rate=(
            BEST_LEARNING_RATE
        ),

        weight_decay=(
            BEST_WEIGHT_DECAY
        ),

        # Canonical batch configuration
        per_device_train_batch_size=(
            TRAIN_BATCH_SIZE
        ),

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        per_device_eval_batch_size=(
            EVAL_BATCH_SIZE
        ),

        # Augmented warmup equivalent
        warmup_steps=(
            BEST_WARMUP_STEPS
        ),

        # Training duration
        num_train_epochs=(
            FINAL_MAX_EPOCHS
        ),

        # Validation-only checkpoint selection
        load_best_model_at_end=True,

        metric_for_best_model=(
            CHECKPOINT_SELECTION_METRIC
        ),

        greater_is_better=True,

        # Reproducibility
        seed=seed,

        data_seed=seed,

        # BF16 only for Trainer mixed precision
        bf16=(
            torch.cuda.is_available()
        ),

        fp16=False,

        report_to="none",

        disable_tqdm=False,
    )


    # ========================================================
    # TRAINING CONFIGURATION AUDIT
    # ========================================================

    print(
        "\nTraining configuration:"
    )

    print(
        "Physical train batch:",
        training_args.per_device_train_batch_size
    )

    print(
        "Gradient accumulation:",
        training_args.gradient_accumulation_steps
    )

    print(
        "Effective batch:",
        (
            training_args.per_device_train_batch_size
            *
            training_args.gradient_accumulation_steps
        )
    )

    print(
        "Evaluation batch:",
        training_args.per_device_eval_batch_size
    )

    print(
        "Warmup steps:",
        training_args.warmup_steps
    )

    print(
        "Trainer BF16:",
        training_args.bf16
    )


    assert (
        training_args.per_device_train_batch_size
        ==
        16
    )

    assert (
        training_args.gradient_accumulation_steps
        ==
        1
    )

    assert (
        training_args.per_device_eval_batch_size
        ==
        16
    )

    assert (
        training_args.warmup_steps
        ==
        185
    )


    # ========================================================
    # TRAINER
    # ========================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=(
            train_unfiltered_aug_dataset
        ),

        eval_dataset=(
            val_dataset
        ),

        data_collator=(
            data_collator
        ),

        compute_metrics=(
            compute_metrics
        ),

        processing_class=(
            tokenizer
        ),

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=(
                    FINAL_EARLY_STOPPING_PATIENCE
                ),

                early_stopping_threshold=0.0,
            )
        ],
    )


    # ========================================================
    # TRAIN
    # ========================================================

    print(
        "\nStarting corrected seed 44 training..."
    )


    start_time = time.time()


    trainer.train()


    training_time = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )


    best_validation_macro_f1 = (
        trainer.state.best_metric
    )


    epoch_reached = (
        trainer.state.epoch
    )


    assert (
        best_checkpoint
        is not None
    )


    best_checkpoint_path = Path(
        best_checkpoint
    )


    assert (
        best_checkpoint_path.exists()
    )


    print(
        "\nBest checkpoint:"
    )

    print(
        best_checkpoint_path
    )


    print(
        "Best validation Macro F1:",
        f"{best_validation_macro_f1:.4f}"
    )


    print(
        "Epoch reached:",
        epoch_reached
    )


    print(
        "Training time:",
        f"{training_time / 60:.2f} minutes"
    )


    # ========================================================
    # VALIDATION PREDICTIONS
    # ========================================================

    prediction_output = (
        trainer.predict(
            val_dataset
        )
    )


    y_true = (
        prediction_output.label_ids
    )


    y_pred = np.argmax(
        prediction_output.predictions,
        axis=1,
    )


    assert (
        len(
            y_true
        )
        ==
        801
    )

    assert (
        len(
            y_pred
        )
        ==
        801
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_true,

        y_pred,

        labels=METRIC_LABELS,

        target_names=[
            id2label[
                class_id
            ]
            for class_id
            in METRIC_LABELS
        ],

        output_dict=True,

        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    print(
        "\nPER-CLASS VALIDATION RESULTS - SEED 44"
    )


    display(
        report_df.round(4)
    )


    # ========================================================
    # STORE OVERALL RESULT
    # ========================================================

    overall_result = {

        "seed":
            seed,

        "best_checkpoint":
            str(
                best_checkpoint_path
            ),

        "epoch_reached":
            epoch_reached,

        "best_validation_macro_f1":
            best_validation_macro_f1,

        "validation_accuracy":
            report[
                "accuracy"
            ],

        "validation_macro_precision":
            report[
                "macro avg"
            ][
                "precision"
            ],

        "validation_macro_recall":
            report[
                "macro avg"
            ][
                "recall"
            ],

        "validation_macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ],

        "validation_weighted_precision":
            report[
                "weighted avg"
            ][
                "precision"
            ],

        "validation_weighted_recall":
            report[
                "weighted avg"
            ][
                "recall"
            ],

        "validation_weighted_f1":
            report[
                "weighted avg"
            ][
                "f1-score"
            ],

        "training_time_minutes":
            training_time / 60,

        "training_data_sha256":
            current_training_hash,

        "model_initial_dtype":
            str(
                model_parameter_dtype
            ),

        "physical_train_batch":
            TRAIN_BATCH_SIZE,

        "gradient_accumulation":
            GRADIENT_ACCUMULATION_STEPS,

        "effective_batch":
            EFFECTIVE_BATCH_SIZE,

        "eval_batch":
            EVAL_BATCH_SIZE,

        "warmup_steps":
            BEST_WARMUP_STEPS,
    }


    unfiltered_overall_results.append(
        overall_result
    )


    # ========================================================
    # STORE PER-CLASS RESULT
    # ========================================================

    for class_id in METRIC_LABELS:

        class_name = (
            id2label[
                class_id
            ]
        )

        class_metrics = (
            report[
                class_name
            ]
        )


        unfiltered_per_class_results.append({

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        })


    # ========================================================
    # SAVE SEED-44 VALIDATION PREDICTIONS
    # ========================================================

    prediction_df = pd.DataFrame({

        "true_label_id":
            y_true,

        "predicted_label_id":
            y_pred,

        "true_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_true
            ],

        "predicted_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_pred
            ],
    })


    prediction_df.to_csv(

        UNFILTERED_RESULTS_DIR
        / "seed_44_validation_predictions.csv",

        index=False,
    )


    # ========================================================
    # SAVE UPDATED THREE-SEED RAW RESULTS
    # ========================================================

    pd.DataFrame(
        unfiltered_overall_results
    ).to_csv(
        overall_results_path,
        index=False,
    )


    pd.DataFrame(
        unfiltered_per_class_results
    ).to_csv(
        per_class_results_path,
        index=False,
    )


    # ========================================================
    # RETAIN CHECKPOINT
    # ========================================================

    assert (
        best_checkpoint_path.exists()
    )


    print(
        "\nKeeping seed 44 checkpoint:"
    )

    print(
        best_checkpoint_path
    )


    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    del prediction_output
    del trainer
    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    assert (
        best_checkpoint_path.exists()
    )


    print(
        "Checkpoint confirmed after cleanup."
    )


# ============================================================
# FINAL THREE-SEED DATAFRAMES
# ============================================================

unfiltered_overall_results_df = (
    pd.DataFrame(
        unfiltered_overall_results
    )
    .sort_values(
        "seed"
    )
    .reset_index(
        drop=True
    )
)


unfiltered_per_class_results_df = (
    pd.DataFrame(
        unfiltered_per_class_results
    )
    .sort_values(
        [
            "seed",
            "class_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# VERIFY EXACTLY 3 SEEDS
# ============================================================

assert (
    unfiltered_overall_results_df[
        "seed"
    ]
    .astype(int)
    .tolist()
    ==
    [
        42,
        43,
        44,
    ]
)


assert (
    len(
        unfiltered_per_class_results_df
    )
    ==
    15
)


# ============================================================
# VERIFY SAME FIXED DATASET ACROSS ALL SEEDS
# ============================================================

assert (
    unfiltered_overall_results_df[
        "training_data_sha256"
    ]
    .nunique()
    ==
    1
)


assert (
    unfiltered_overall_results_df[
        "training_data_sha256"
    ]
    .iloc[0]
    ==
    UNFILTERED_TRAINING_DATA_HASH
)


print("\n" + "=" * 80)
print(
    "CROSS-SEED TRAINING-DATA CONSISTENCY: PASSED"
)
print("=" * 80)


# ============================================================
# OVERALL THREE-SEED SUMMARY
# ============================================================

summary_columns = [
    "validation_accuracy",
    "validation_macro_precision",
    "validation_macro_recall",
    "validation_macro_f1",
    "validation_weighted_f1",
]


unfiltered_overall_summary_df = (

    unfiltered_overall_results_df[
        summary_columns
    ]

    .agg(
        [
            "mean",
            "std",
        ]
    )
)


print(
    "\nCORRECTED UNFILTERED 1x - THREE-SEED RESULTS:"
)


display(
    unfiltered_overall_results_df.round(4)
)


print(
    "\nThree-seed validation summary:"
)


display(
    unfiltered_overall_summary_df.round(4)
)


mean_validation_macro_f1 = (
    unfiltered_overall_results_df[
        "validation_macro_f1"
    ]
    .mean()
)


std_validation_macro_f1 = (
    unfiltered_overall_results_df[
        "validation_macro_f1"
    ]
    .std(
        ddof=1
    )
)


print(
    "\nValidation Macro F1:"
)

print(
    f"{mean_validation_macro_f1:.4f}"
    " ± "
    f"{std_validation_macro_f1:.4f}"
)


# ============================================================
# PER-CLASS THREE-SEED SUMMARY
# ============================================================

unfiltered_per_class_summary_df = (

    unfiltered_per_class_results_df

    .groupby(
        [
            "class_id",
            "class_name",
        ],
        as_index=False,
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )
)


print(
    "\nThree-seed per-class validation summary:"
)


display(
    unfiltered_per_class_summary_df.round(4)
)


# ============================================================
# SAVE FINAL THREE-SEED SUMMARIES
# ============================================================

unfiltered_overall_results_df.to_csv(
    overall_results_path,
    index=False,
)


unfiltered_per_class_results_df.to_csv(
    per_class_results_path,
    index=False,
)


unfiltered_overall_summary_df.to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_overall_summary.csv"
)


unfiltered_per_class_summary_df.to_csv(

    UNFILTERED_RESULTS_DIR
    / "unfiltered_1x_corrected_3seed_per_class_summary.csv",

    index=False,
)


# ============================================================
# FINAL CHECKPOINT COLLECTION
# ============================================================

seed44_dir = (
    UNFILTERED_OUTPUT_DIR
    / "seed_44"
)


seed44_checkpoints = sorted(
    seed44_dir.glob(
        "checkpoint-*"
    )
)


assert (
    len(
        seed44_checkpoints
    )
    ==
    1
), (
    f"Expected exactly one seed-44 checkpoint; "
    f"found {len(seed44_checkpoints)}."
)


SEED44_CHECKPOINT = (
    seed44_checkpoints[0]
)


FINAL_UNFILTERED_CHECKPOINTS = {

    42:
        SEED42_CHECKPOINT,

    43:
        SEED43_CHECKPOINT,

    44:
        SEED44_CHECKPOINT,
}


# ============================================================
# FINAL CHECKPOINT VERIFICATION
# ============================================================

print("\n" + "=" * 80)

print(
    "CORRECTED UNFILTERED CHECKPOINTS READY FOR FINAL TEST"
)

print("=" * 80)


for seed, checkpoint in (
    FINAL_UNFILTERED_CHECKPOINTS.items()
):

    assert checkpoint.exists()

    print(
        f"Seed {seed}:",
        checkpoint
    )


print(
    "\nAll corrected unfiltered results saved to:"
)

print(
    UNFILTERED_RESULTS_DIR
)

CORRECTED UNFILTERED FINAL CONTINUATION
Seed 42 checkpoint:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42/checkpoint-4419

Seed 43 checkpoint:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_43/checkpoint-1964

Remaining seed: [44]

Configuration verification: PASSED

Free disk space: 16.72 GB
Disk-space check: PASSED

Existing stored seeds: [42, 43]
Existing result-state verification: PASSED


CORRECTED UNFILTERED 1x XLM-R - SEED 44
Training-data hash: e3a56bdbb7d79ae0bf34d0fd9566c1f4d016bdcc2866d6ee07362043c5116049
Fixed dataset integrity: PASSED


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[HAMI-core Msg(1343:140007575660352:multiprocess_memory_limit.c:455)]: Calling exit


Model parameter dtype before Trainer: torch.float32
FP32 pretrained model loading: VERIFIED

Training configuration:
Physical train batch: 16
Gradient accumulation: 1
Effective batch: 16
Evaluation batch: 16
Warmup steps: 185
Trainer BF16: True

Starting corrected seed 44 training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,1.242678,0.827442,0.710362,0.615765,0.684149,0.618257,0.728037,0.710362,0.693718
2,0.869825,0.895495,0.689139,0.636442,0.683111,0.642764,0.743844,0.689139,0.701001
3,0.701370,0.658002,0.786517,0.717239,0.705153,0.704716,0.785360,0.786517,0.781409
4,0.573677,0.839451,0.729089,0.623424,0.720518,0.634410,0.771140,0.729089,0.723905
5,0.465634,0.735368,0.782772,0.682219,0.741464,0.704708,0.793600,0.782772,0.784078


...


Three-seed validation summary:


,validation_accuracy,validation_macro_precision,validation_macro_recall,validation_macro_f1,validation_weighted_f1
mean,0.7799,0.6963,0.7189,0.6990,0.7778
std,0.0063,0.0206,0.0220,0.0074,0.0068



Validation Macro F1:
0.6990 ± 0.0074

Three-seed per-class validation summary:


,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support
0,0,Abusive/Offensive,0.7498,0.0077,0.5885,0.0325,0.6592,0.0236,192.0
1,1,Normal,0.8735,0.0253,0.8949,0.0292,0.8835,0.0017,428.0
2,2,Religious Hate,0.6617,0.0540,0.6349,0.1111,0.6411,0.0270,63.0
3,3,Sexism,0.5944,0.0424,0.8358,0.0299,0.6936,0.0237,67.0
4,4,Profane,0.6020,0.0963,0.6405,0.0113,0.6177,0.0522,51.0



CORRECTED UNFILTERED CHECKPOINTS READY FOR FINAL TEST
Seed 42: /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_42/checkpoint-4419
Seed 43: /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_43/checkpoint-1964
Seed 44: /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected/seed_44/checkpoint-1473

All corrected unfiltered results saved to:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_corrected_results


In [28]:
# ============================================================
# FINAL TEST EVALUATION
# UNFILTERED 1x AUGMENTED XLM-R
# THREE RETAINED VALIDATION-SELECTED CHECKPOINTS
#
# IMPORTANT:
# - Test set is used ONLY for final evaluation.
# - Checkpoints were selected using validation Macro F1.
# - All 3 seeds are evaluated independently.
# - Raw predictions, per-class metrics, confusion matrices,
#   and overall mean ± standard deviation are saved.
# ============================================================

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from transformers import (
    AutoModelForSequenceClassification,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


# ============================================================
# RETAINED BEST VALIDATION-SELECTED CHECKPOINTS
# ============================================================

UNFILTERED_TEST_CHECKPOINTS = {

    42: Path(
        "/home/jovyan/project work/data_analyssis/"
        "classifier/outputs/xlm_roberta_unfiltered_1x/"
        "seed_42/checkpoint-3928"
    ),

    43: Path(
        "/home/jovyan/project work/data_analyssis/"
        "classifier/outputs/xlm_roberta_unfiltered_1x/"
        "seed_43/checkpoint-3437"
    ),

    44: Path(
        "/home/jovyan/project work/data_analyssis/"
        "classifier/outputs/xlm_roberta_unfiltered_1x/"
        "seed_44/checkpoint-2946"
    ),
}


# ============================================================
# CHECKPOINT INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("UNFILTERED 1x FINAL TEST CHECKPOINTS")
print("=" * 70)

for seed, checkpoint in (
    UNFILTERED_TEST_CHECKPOINTS.items()
):

    print(
        f"Seed {seed}:",
        checkpoint.exists(),
        checkpoint
    )

    assert checkpoint.exists(), (
        f"Checkpoint missing for seed {seed}: "
        f"{checkpoint}"
    )

    assert (
        (checkpoint / "model.safetensors").exists()
        or
        (checkpoint / "pytorch_model.bin").exists()
    ), (
        f"Model weights missing for seed {seed}"
    )

    assert (
        checkpoint / "config.json"
    ).exists(), (
        f"config.json missing for seed {seed}"
    )


print(
    "\nAll three retained checkpoints verified."
)


# ============================================================
# FINAL TEST-SET INTEGRITY CHECK
# ============================================================

assert len(test_df) == 2003
assert len(test_dataset) == 2003

assert (
    test_df["tweet"]
    .isna()
    .sum()
    == 0
)

assert (
    test_df["label"]
    .isna()
    .sum()
    == 0
)


EXPECTED_TEST_CLASS_COUNTS = {
    0: 481,
    1: 1070,
    2: 156,
    3: 168,
    4: 128,
}


actual_test_class_counts = (
    test_df[
        "label"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_test_class_counts
    ==
    EXPECTED_TEST_CLASS_COUNTS
), (
    "Unexpected test class distribution.\n"
    f"Expected: {EXPECTED_TEST_CLASS_COUNTS}\n"
    f"Actual:   {actual_test_class_counts}"
)


print("\n" + "=" * 70)
print("FINAL TEST-SET INTEGRITY CHECK")
print("=" * 70)

print(
    "Test samples:",
    len(test_df)
)

print(
    "Missing tweets:",
    test_df["tweet"].isna().sum()
)

print(
    "Missing labels:",
    test_df["label"].isna().sum()
)

print(
    "\nTest class distribution:"
)

display(
    test_df[
        "label"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

print(
    "\nTest-set integrity check: PASSED"
)


# ============================================================
# TEST DATALOADER
#
# shuffle=False is CRITICAL:
# preserves exact test-set order.
#
# Small inference batch only reduces memory usage.
# It does NOT alter deterministic predictions.
# ============================================================

TEST_BATCH_SIZE = 4


test_loader = DataLoader(

    test_dataset,

    batch_size=TEST_BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,
)


print(
    "\nTest DataLoader samples:",
    len(test_dataset)
)

print(
    "Test inference batch size:",
    TEST_BATCH_SIZE
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

UNFILTERED_TEST_RESULTS_DIR = (
    UNFILTERED_RESULTS_DIR
    / "final_test"
)


UNFILTERED_TEST_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "\nFinal test results directory:"
)

print(
    UNFILTERED_TEST_RESULTS_DIR
)


# ============================================================
# RESULT STORAGE
# ============================================================

unfiltered_test_overall_results = []

unfiltered_test_per_class_results = []

seed_test_labels = {}


# ============================================================
# THREE-SEED FINAL TEST LOOP
# ============================================================

for seed, checkpoint_path in (
    UNFILTERED_TEST_CHECKPOINTS.items()
):

    print("\n")
    print("=" * 80)

    print(
        f"UNFILTERED 1x FINAL TEST - SEED {seed}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Cleanup before loading checkpoint
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    print(
        "Evaluation device:",
        device
    )


    # ========================================================
    # LOAD VALIDATION-SELECTED CHECKPOINT
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            checkpoint_path,

            dtype=(
                torch.bfloat16
                if torch.cuda.is_available()
                else torch.float32
            ),
        )
    )


    model = model.to(
        device
    )


    model.eval()


    print(
        "Loaded checkpoint:"
    )

    print(
        checkpoint_path
    )

    print(
        "Model device:",
        next(
            model.parameters()
        ).device
    )

    print(
        "Model dtype:",
        next(
            model.parameters()
        ).dtype
    )


    # ========================================================
    # DIRECT TEST INFERENCE
    # ========================================================

    all_predictions = []

    all_labels = []


    with torch.inference_mode():

        for batch in test_loader:

            labels = batch.pop(
                "labels"
            )


            batch = {

                key:
                    value.to(
                        device
                    )

                for key, value
                in batch.items()
            }


            outputs = model(
                **batch
            )


            predictions = (

                outputs.logits

                .argmax(
                    dim=-1
                )

                .detach()

                .cpu()

                .numpy()
            )


            all_predictions.extend(
                predictions
            )


            all_labels.extend(
                labels
                .cpu()
                .numpy()
            )


    # ========================================================
    # CONVERT TO ARRAYS
    # ========================================================

    y_true = np.asarray(
        all_labels,
        dtype=int,
    )


    y_pred = np.asarray(
        all_predictions,
        dtype=int,
    )


    # ========================================================
    # SEED-LEVEL TEST INTEGRITY CHECK
    # ========================================================

    assert len(y_true) == 2003

    assert len(y_pred) == 2003


    assert set(
        np.unique(
            y_true
        )
    ).issubset(
        {
            0,
            1,
            2,
            3,
            4,
        }
    )


    assert set(
        np.unique(
            y_pred
        )
    ).issubset(
        {
            0,
            1,
            2,
            3,
            4,
        }
    )


    # --------------------------------------------------------
    # Ensure DataLoader test labels are in exactly the same
    # order as the untouched test dataframe.
    # --------------------------------------------------------

    expected_test_labels = (
        test_df[
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    assert np.array_equal(
        y_true,
        expected_test_labels
    ), (
        f"Test-label order mismatch for seed {seed}"
    )


    # --------------------------------------------------------
    # Save labels for cross-seed consistency verification.
    # --------------------------------------------------------

    seed_test_labels[
        seed
    ] = y_true.copy()


    print(
        f"\nSeed {seed} test integrity check: PASSED"
    )

    print(
        "Test predictions completed:",
        len(y_pred)
    )


    # ========================================================
    # OVERALL TEST METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        y_true,
        y_pred,
    )


    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
            4,
        ],

        average="macro",

        zero_division=0,
    )


    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = precision_recall_fscore_support(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
            4,
        ],

        average="weighted",

        zero_division=0,
    )


    print("\n")
    print("-" * 60)

    print(
        f"FINAL TEST RESULTS - SEED {seed}"
    )

    print("-" * 60)

    print(
        f"Accuracy        : "
        f"{test_accuracy:.4f}"
    )

    print(
        f"Macro Precision : "
        f"{macro_precision:.4f}"
    )

    print(
        f"Macro Recall    : "
        f"{macro_recall:.4f}"
    )

    print(
        f"Macro F1        : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1     : "
        f"{weighted_f1:.4f}"
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
            4,
        ],

        target_names=[
            "Abusive/Offensive",
            "Normal",
            "Religious Hate",
            "Sexism",
            "Profane",
        ],

        output_dict=True,

        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    print(
        f"\nPer-class test results - seed {seed}:"
    )


    display(
        report_df.round(4)
    )


    # ========================================================
    # SAVE PER-CLASS REPORT
    # ========================================================

    report_df.to_csv(

        UNFILTERED_TEST_RESULTS_DIR
        / (
            f"seed_{seed}_"
            "test_per_class.csv"
        )
    )


    # ========================================================
    # STORE PER-CLASS RESULTS
    # ========================================================

    for class_id, class_name in (
        id2label.items()
    ):

        class_metrics = (
            report[
                class_name
            ]
        )


        unfiltered_test_per_class_results.append({

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        })


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
            4,
        ],
    )


    # --------------------------------------------------------
    # Verify confusion matrix contains exactly 2003 examples.
    # --------------------------------------------------------

    assert (
        cm.sum()
        == 2003
    )


    cm_df = pd.DataFrame(

        cm,

        index=[
            id2label[i]
            for i in range(5)
        ],

        columns=[
            id2label[i]
            for i in range(5)
        ],
    )


    cm_df.to_csv(

        UNFILTERED_TEST_RESULTS_DIR
        / (
            f"seed_{seed}_"
            "test_confusion_matrix.csv"
        )
    )


    # ========================================================
    # SAVE RAW TEST PREDICTIONS
    # ========================================================

    prediction_df = pd.DataFrame({

        "test_index":
            np.arange(
                len(test_df)
            ),

        "tweet":
            test_df[
                "tweet"
            ].values,

        "true_label_id":
            y_true,

        "predicted_label_id":
            y_pred,

        "true_label":
            [
                id2label[
                    int(label)
                ]
                for label in y_true
            ],

        "predicted_label":
            [
                id2label[
                    int(label)
                ]
                for label in y_pred
            ],

        "correct":
            (
                y_true
                ==
                y_pred
            ),
    })


    # --------------------------------------------------------
    # Final raw prediction integrity check.
    # --------------------------------------------------------

    assert len(
        prediction_df
    ) == 2003


    assert (
        prediction_df[
            "test_index"
        ]
        .duplicated()
        .sum()
        == 0
    )


    assert (
        prediction_df[
            "true_label_id"
        ]
        .astype(int)
        .to_numpy()
        ==
        expected_test_labels
    ).all()


    prediction_df.to_csv(

        UNFILTERED_TEST_RESULTS_DIR
        / (
            f"seed_{seed}_"
            "test_predictions.csv"
        ),

        index=False,

        encoding="utf-8",
    )


    # ========================================================
    # STORE OVERALL RESULT
    # ========================================================

    unfiltered_test_overall_results.append({

        "seed":
            seed,

        "checkpoint":
            str(
                checkpoint_path
            ),

        "test_samples":
            len(
                y_true
            ),

        "test_accuracy":
            test_accuracy,

        "test_macro_precision":
            macro_precision,

        "test_macro_recall":
            macro_recall,

        "test_macro_f1":
            macro_f1,

        "test_weighted_precision":
            weighted_precision,

        "test_weighted_recall":
            weighted_recall,

        "test_weighted_f1":
            weighted_f1,
    })


    # ========================================================
    # SAVE INCREMENTALLY AFTER EVERY SEED
    # ========================================================

    pd.DataFrame(
        unfiltered_test_overall_results
    ).to_csv(

        UNFILTERED_TEST_RESULTS_DIR
        / "unfiltered_1x_test_seed_results.csv",

        index=False,
    )


    # ========================================================
    # GPU CLEANUP BEFORE NEXT SEED
    # ========================================================

    del outputs

    del model


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        f"\nSeed {seed} final test evaluation safely saved."
    )


# ============================================================
# CROSS-SEED TEST-LABEL CONSISTENCY CHECK
# ============================================================

reference_labels = (
    seed_test_labels[
        42
    ]
)


for seed in [
    43,
    44,
]:

    assert np.array_equal(
        reference_labels,
        seed_test_labels[
            seed
        ]
    ), (
        f"Test labels differ between seed 42 and seed {seed}"
    )


print("\n" + "=" * 70)

print(
    "CROSS-SEED TEST-LABEL CONSISTENCY: PASSED"
)

print("=" * 70)


# ============================================================
# THREE-SEED OVERALL TEST RESULTS
# ============================================================

unfiltered_test_overall_df = (
    pd.DataFrame(
        unfiltered_test_overall_results
    )
)


assert len(
    unfiltered_test_overall_df
) == 3


assert set(
    unfiltered_test_overall_df[
        "seed"
    ]
) == {
    42,
    43,
    44,
}


print("\n")
print("=" * 80)

print(
    "UNFILTERED 1x - FINAL THREE-SEED TEST RESULTS"
)

print("=" * 80)


display(
    unfiltered_test_overall_df.round(4)
)


# ============================================================
# THREE-SEED MEAN ± STANDARD DEVIATION
# ============================================================

summary_columns = [

    "test_accuracy",

    "test_macro_precision",

    "test_macro_recall",

    "test_macro_f1",

    "test_weighted_f1",
]


unfiltered_test_summary_df = (

    unfiltered_test_overall_df[
        summary_columns
    ]

    .agg(
        [
            "mean",
            "std",
        ]
    )
)


print(
    "\nThree-seed final test summary:"
)


display(
    unfiltered_test_summary_df.round(4)
)


print(
    "\nFINAL TEST MACRO F1:"
)


print(
    f'{unfiltered_test_overall_df["test_macro_f1"].mean():.4f}'
    " ± "
    f'{unfiltered_test_overall_df["test_macro_f1"].std(ddof=1):.4f}'
)


# ============================================================
# THREE-SEED PER-CLASS SUMMARY
# ============================================================

unfiltered_test_per_class_df = (
    pd.DataFrame(
        unfiltered_test_per_class_results
    )
)


assert len(
    unfiltered_test_per_class_df
) == 15


unfiltered_test_per_class_summary_df = (

    unfiltered_test_per_class_df

    .groupby(
        [
            "class_id",
            "class_name",
        ]
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )

    .reset_index()
)


print(
    "\nThree-seed per-class FINAL TEST summary:"
)


display(
    unfiltered_test_per_class_summary_df.round(4)
)


# ============================================================
# SAVE FINAL TEST SUMMARIES
# ============================================================

unfiltered_test_overall_df.to_csv(

    UNFILTERED_TEST_RESULTS_DIR
    / "unfiltered_1x_test_all_seed_results.csv",

    index=False,
)


unfiltered_test_summary_df.to_csv(

    UNFILTERED_TEST_RESULTS_DIR
    / "unfiltered_1x_test_overall_summary.csv"
)


unfiltered_test_per_class_df.to_csv(

    UNFILTERED_TEST_RESULTS_DIR
    / "unfiltered_1x_test_per_class_all_seeds.csv",

    index=False,
)


unfiltered_test_per_class_summary_df.to_csv(

    UNFILTERED_TEST_RESULTS_DIR
    / "unfiltered_1x_test_per_class_summary.csv",

    index=False,
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 80)

print(
    "UNFILTERED 1x FINAL TEST EVALUATION COMPLETE"
)

print("=" * 80)

print(
    "Seeds evaluated:",
    sorted(
        unfiltered_test_overall_df[
            "seed"
        ].tolist()
    )
)

print(
    "Test samples per seed:",
    unfiltered_test_overall_df[
        "test_samples"
    ].tolist()
)

print(
    "\nMean test Macro F1:",
    f'{unfiltered_test_overall_df["test_macro_f1"].mean():.4f}'
)

print(
    "Test Macro F1 standard deviation:",
    f'{unfiltered_test_overall_df["test_macro_f1"].std(ddof=1):.4f}'
)

print(
    "\nAll final unfiltered test results saved to:"
)

print(
    UNFILTERED_TEST_RESULTS_DIR
)

UNFILTERED 1x FINAL TEST CHECKPOINTS
Seed 42: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x/seed_42/checkpoint-3928
Seed 43: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x/seed_43/checkpoint-3437
Seed 44: True /home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x/seed_44/checkpoint-2946

All three retained checkpoints verified.

FINAL TEST-SET INTEGRITY CHECK
Test samples: 2003
Missing tweets: 0
Missing labels: 0

Test class distribution:


,label,count
0,0,481
1,1,1070
2,2,156
3,3,168
4,4,128



Test-set integrity check: PASSED

Test DataLoader samples: 2003
Test inference batch size: 4

Final test results directory:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_results/final_test


UNFILTERED 1x FINAL TEST - SEED 42
Evaluation device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded checkpoint:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x/seed_42/checkpoint-3928
Model device: cuda:0
Model dtype: torch.bfloat16

Seed 42 test integrity check: PASSED
Test predictions completed: 2003


------------------------------------------------------------
FINAL TEST RESULTS - SEED 42
------------------------------------------------------------
Accuracy        : 0.7064
Macro Precision : 0.5883
Macro Recall    : 0.6023
Macro F1        : 0.5905
Weighted F1     : 0.6997

Per-class test results - seed 42:


...


Three-seed final test summary:


,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1
mean,0.6966,0.5732,0.5843,0.5745,0.6901
std,0.0093,0.0137,0.0157,0.0140,0.0089



FINAL TEST MACRO F1:
0.5745 ± 0.0140

Three-seed per-class FINAL TEST summary:


,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support
0,0,Abusive/Offensive,0.5917,0.0317,0.4345,0.0200,0.5010,0.0241,481.0
1,1,Normal,0.8274,0.0059,0.8835,0.0047,0.8545,0.0012,1070.0
2,2,Religious Hate,0.5156,0.0164,0.6325,0.0329,0.5675,0.0055,156.0
3,3,Sexism,0.4016,0.0622,0.4504,0.0481,0.4244,0.0559,168.0
4,4,Profane,0.5297,0.0181,0.5208,0.0090,0.5250,0.0065,128.0



UNFILTERED 1x FINAL TEST EVALUATION COMPLETE
Seeds evaluated: [42, 43, 44]
Test samples per seed: [2003, 2003, 2003]

Mean test Macro F1: 0.5745
Test Macro F1 standard deviation: 0.0140

All final unfiltered test results saved to:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_unfiltered_1x_results/final_test
